In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA = ROOT / "data"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

model_file = next(
    p for p in DATA.glob("*.xlsx")
    if "v0.5" in p.name
    and "FIXED" in p.name
    and not p.name.startswith("~$")
)

print("MODEL FILE:", model_file)

print("ROOT:", ROOT)
print("MODEL:", model_file.name)

MODEL FILE: /Users/bensfolderaccount/Desktop/FPL 26-27/data/FPL_26_27_Model_v0.5_Team_Fixture_Engine_FIXED.xlsx
ROOT: /Users/bensfolderaccount/Desktop/FPL 26-27
MODEL: FPL_26_27_Model_v0.5_Team_Fixture_Engine_FIXED.xlsx


In [2]:
team_hist = pd.read_csv(
    DATA / "team_strength_25_26.csv"
)

print(
    "Cached team strengths:",
    team_hist.shape,
    "| Teams:",
    team_hist["Team"].nunique(),
)

Cached team strengths: (20, 8) | Teams: 20


In [3]:
from src.data_loader import load_model
from src.fpl_api import fetch_current_players
from src.lineup_sources import (
    fetch_lineup_sources,
    parse_ffscout,
    parse_rotowire,
)
from src.lineup_consensus import (
    match_source_to_players,
    build_consensus,
)
from src.start_probs import build_start_probs
from src.minutes import build_expected_minutes

In [4]:
model = load_model(model_file)

current_players = fetch_current_players()

print("Workbook players:", model["Players"].shape)
print("Live FPL players:", current_players.shape)

/Users/bensfolderaccount/Desktop/FPL 26-27/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/bensfolderaccount/Desktop/FPL 26-27/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


Workbook players: (567, 24)
Live FPL players: (599, 13)


In [5]:
import requests
import re
from bs4 import BeautifulSoup

NMA_URL = "https://www.nevermanagealone.com/playerpicks/16313/fpl-gameweek-1-predicted-lineups-every-premier-league-starting-xi?"

html = requests.get(
    NMA_URL,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30,
).text

soup = BeautifulSoup(html, "lxml")

text = soup.get_text("\n", strip=True)

print("HTML:", len(html))
print("Arsenal:", "Arsenal predicted line-up" in text)
print("Raya:", "Raya" in text)
print("Haaland:", "Haaland" in text)
print("Chelsea predicted:", "Chelsea predicted line-up" in text)

HTML: 378984
Arsenal: True
Raya: True
Haaland: True
Chelsea predicted: True


In [6]:
import re
import pandas as pd

lines = list(soup.stripped_strings)

rows = []

heading_re = re.compile(
    r"^(.+?) predicted line-up(?:\s+\([^)]+\))?:$",
    re.I,
)

for i, line in enumerate(lines):

    m = heading_re.match(line)

    if not m:
        continue

    team = m.group(1).strip()

    if i + 1 >= len(lines):
        continue

    lineup_text = lines[i + 1].strip()

    players = [
        p.strip().rstrip(".")
        for p in re.split(r"[;,]", lineup_text)
        if p.strip()
    ]

    print(team, len(players), players)

    for player in players:
        rows.append({
            "Source": "NMA",
            "Team": team,
            "Player Raw": player,
            "Predicted Starter": True,
        })

nma = pd.DataFrame(rows)

print("\nShape:", nma.shape)
display(nma[nma["Team"] == "Arsenal"])

Arsenal 11 ['Raya', 'White', 'Mosquera', 'Gabriel', 'Calafiori', 'Odegaard', 'Lewis-Skelly', 'Rice', 'Madueke', 'Gyokeres', 'Tzolis']
Coventry City 11 ['Rushworth', 'van Ewijk', 'Thomas', 'Amenda', 'Jay da Silva', 'Yirenkyi', 'Grimes', 'Onyeka', 'Tchaouna', 'Simms', 'Thomas-Asante']
Hull City 11 ['Tzolakis', 'Coyle', 'Ajayi', 'Mendy', 'Giles', 'Slater', 'Crooks', 'Belloumi', 'Hjertø-Dahl', 'Stroud', 'McBurnie']
Manchester United 11 ['Lammens', 'Dalot', 'Maguire', 'Heaven', 'Shaw', 'Tielemans', 'Andrey Santos', 'Amad Diallo', 'Bruno Fernandes', 'Dorgu', 'Mbeumo']
Ipswich Town 11 ['Scherpen', 'O’Shea', 'Diop', 'Greaves', 'Davis', 'Núñez', 'Lukic', 'Fatawu', 'Egeli', 'Maeda', 'Emersonn']
Sunderland 11 ['Roefs', 'Meunier', 'O’Nien', 'Ballard', 'Reinildo', 'Xhaka', 'Sadiki', 'Hume', 'Le Fée', 'Angulo', 'Brobbey']
Everton 11 ['Pickford', 'O’Brien', 'Tarkowski', 'Branthwaite', 'Mykolenko', 'Hackney', 'Armstrong', 'Röhl', 'Dewsbury-Hall', 'Ndiaye', 'Barry']
Crystal Palace 11 ['Henderson', 'Can

,Source,Team,Player Raw,Predicted Starter
0,NMA,Arsenal,Raya,True
1,NMA,Arsenal,White,True
2,NMA,Arsenal,Mosquera,True
3,NMA,Arsenal,Gabriel,True
4,NMA,Arsenal,Calafiori,True
5,NMA,Arsenal,Odegaard,True
6,NMA,Arsenal,Lewis-Skelly,True
7,NMA,Arsenal,Rice,True
8,NMA,Arsenal,Madueke,True
9,NMA,Arsenal,Gyokeres,True


In [7]:
nma_match = match_source_to_players(
    nma,
    current_players,
)

print(
    "NMA unmatched:",
    nma_match["Player ID"].isna().sum(),
)

display(
    nma_match[
        nma_match["Player ID"].isna()
    ][
        [
            "Source",
            "Team",
            "Matched Team",
            "Player Raw",
            "Match Score",
        ]
    ]
)

NMA unmatched: 0


,Source,Team,Matched Team,Player Raw,Match Score


In [8]:
raw_lineups = fetch_lineup_sources(ROOT)

raw_lineups

{'ffscout': PosixPath('/Users/bensfolderaccount/Desktop/FPL 26-27/data/raw/lineups/2026-08-21_2147/ffscout.html'),
 'rotowire': PosixPath('/Users/bensfolderaccount/Desktop/FPL 26-27/data/raw/lineups/2026-08-21_2147/rotowire.html'),
 'nma': PosixPath('/Users/bensfolderaccount/Desktop/FPL 26-27/data/raw/lineups/2026-08-21_2147/nma.html')}

In [9]:
from src.lineup_sources import (
    parse_ffscout,
    parse_rotowire,
    parse_nma,
)

ffs = parse_ffscout(
    raw_lineups["ffscout"]
)

rw = parse_rotowire(
    raw_lineups["rotowire"]
)

nma = parse_nma(
    raw_lineups["nma"]
)

print(
    "FFScout:",
    ffs.shape,
)

print(
    "RotoWire:",
    rw.shape,
)

print(
    "NMA:",
    nma.shape,
)

FFScout: (220, 4)
RotoWire: (220, 6)
NMA: (220, 4)


In [10]:
ffs_match = match_source_to_players(
    ffs,
    current_players,
)

rw_match = match_source_to_players(
    rw,
    current_players,
)

nma_match = match_source_to_players(
    nma,
    current_players,
)

print(
    "FFScout unmatched:",
    ffs_match["Player ID"].isna().sum(),
)

print(
    "RotoWire unmatched:",
    rw_match["Player ID"].isna().sum(),
)

print(
    "NMA unmatched:",
    nma_match["Player ID"].isna().sum(),
)

FFScout unmatched: 0
RotoWire unmatched: 0
NMA unmatched: 0


In [11]:
consensus = build_consensus(
    current_players,
    ffs_match,
    rw_match,
    nma_match,
)

In [12]:
consensus[
    consensus["Team"] == "Arsenal"
][
    [
        "Player",
        "FFScout",
        "RotoWire",
        "NMA",
        "Lineup Votes",
        "Lineup Consensus",
    ]
]

,Player,FFScout,RotoWire,NMA,Lineup Votes,Lineup Consensus
0,Raya,1.0,1.0,1.0,3.0,1.000000
1,Arrizabalaga,0.0,0.0,0.0,0.0,0.000000
2,Meslier,0.0,0.0,0.0,0.0,0.000000
3,Gabriel,1.0,1.0,1.0,3.0,1.000000
4,J.Timber,0.0,0.0,0.0,0.0,0.000000
5,Saliba,0.0,0.0,0.0,0.0,0.000000
6,Lewis-Skelly,1.0,1.0,1.0,3.0,1.000000
7,Calafiori,1.0,1.0,1.0,3.0,1.000000
8,Hincapie,0.0,0.0,0.0,0.0,0.000000
9,White,1.0,1.0,1.0,3.0,1.000000


In [13]:
consensus.groupby(
    "Lineup Votes"
).size()

Lineup Votes
0.0    358
1.0     23
2.0     17
3.0    201
dtype: int64

In [14]:
consensus.groupby("Team")[
    ["FFScout", "RotoWire", "NMA"]
].sum()

,FFScout,RotoWire,NMA
Team,,,
Arsenal,11.0,11.0,11.0
Aston Villa,11.0,11.0,11.0
Bournemouth,11.0,11.0,11.0
Brentford,11.0,11.0,11.0
Brighton,11.0,11.0,11.0
Chelsea,11.0,11.0,11.0
Coventry City,11.0,11.0,11.0
Crystal Palace,11.0,11.0,11.0
Everton,11.0,11.0,11.0


In [15]:
start_probs = build_start_probs(
    consensus
)

In [16]:
print(start_probs.columns.tolist())
print(start_probs.shape)

['Player ID', 'Code', 'Player', 'Team', 'FPL Pos', 'Status', 'Chance Play Next', 'News', 'FFScout', 'RotoWire', 'NMA', 'Lineup Votes', 'Lineup Consensus', 'Availability Prob', 'Lineup Evidence', 'Conditional Start Prob', 'Source Start Prob', 'Start Prob']
(599, 18)


In [17]:
start_probs[
    start_probs["Team"] == "Arsenal"
][
    [
        "Player",
        "FFScout",
        "RotoWire",
        "NMA",
        "Source Start Prob",
        "Availability Prob",
        "Start Prob",
    ]
].sort_values(
    "Source Start Prob",
    ascending=False,
)

,Player,FFScout,RotoWire,NMA,Source Start Prob,Availability Prob,Start Prob
0,Raya,1.0,1.0,1.0,0.947801,1.00,0.947801
6,Lewis-Skelly,1.0,1.0,1.0,0.947801,1.00,0.947801
12,Rice,1.0,1.0,1.0,0.947801,1.00,0.947801
10,Mosquera,1.0,1.0,1.0,0.947801,1.00,0.947801
9,White,1.0,1.0,1.0,0.947801,1.00,0.947801
7,Calafiori,1.0,1.0,1.0,0.947801,1.00,0.947801
14,Ødegaard,1.0,1.0,1.0,0.947801,1.00,0.947801
27,Tzolis,1.0,1.0,1.0,0.947801,1.00,0.947801
3,Gabriel,1.0,1.0,1.0,0.947801,1.00,0.947801
23,Gyökeres,1.0,0.0,1.0,0.710761,1.00,0.710761


In [18]:
start_probs.groupby("Team")[
    "Source Start Prob"
].sum()

Team
Arsenal           11.096906
Aston Villa       11.447791
Bournemouth       11.366927
Brentford         11.099101
Brighton          11.319327
Chelsea           11.120913
Coventry City     11.113817
Crystal Palace    11.322572
Everton           11.135104
Fulham            11.099476
Hull City         11.569174
Ipswich Town      11.155325
Leeds             11.415629
Liverpool         11.326720
Man City          11.285907
Man Utd           11.251144
Newcastle         11.139013
Nott'm Forest     11.081177
Spurs             11.283213
Sunderland        11.196673
Name: Source Start Prob, dtype: float64

In [19]:
start_probs[
    start_probs["Team"] == "Arsenal"
][
    [
        "Player",
        "Status",
        "Chance Play Next",
        "FFScout",
        "RotoWire",
        "NMA",
        "Source Start Prob",
        "Availability Prob",
        "Start Prob",
    ]
].sort_values(
    "Source Start Prob",
    ascending=False,
)

,Player,Status,Chance Play Next,FFScout,RotoWire,NMA,Source Start Prob,Availability Prob,Start Prob
0,Raya,a,NaN,1.0,1.0,1.0,0.947801,1.00,0.947801
6,Lewis-Skelly,a,NaN,1.0,1.0,1.0,0.947801,1.00,0.947801
12,Rice,a,NaN,1.0,1.0,1.0,0.947801,1.00,0.947801
10,Mosquera,a,NaN,1.0,1.0,1.0,0.947801,1.00,0.947801
9,White,a,100.0,1.0,1.0,1.0,0.947801,1.00,0.947801
7,Calafiori,a,NaN,1.0,1.0,1.0,0.947801,1.00,0.947801
14,Ødegaard,a,NaN,1.0,1.0,1.0,0.947801,1.00,0.947801
27,Tzolis,a,NaN,1.0,1.0,1.0,0.947801,1.00,0.947801
3,Gabriel,a,NaN,1.0,1.0,1.0,0.947801,1.00,0.947801
23,Gyökeres,a,NaN,1.0,0.0,1.0,0.710761,1.00,0.710761


In [20]:
start_probs[
    start_probs["Availability Prob"] < 1
][
    [
        "Player",
        "Team",
        "Status",
        "Chance Play Next",
        "News",
        "Source Start Prob",
        "Availability Prob",
        "Start Prob",
    ]
].sort_values(
    "Availability Prob"
)

,Player,Team,Status,Chance Play Next,News,Source Start Prob,Availability Prob,Start Prob
4,J.Timber,Arsenal,i,0.0,Groin injury - Unknown return date,0.043069,0.00,0.000000
399,Leoni,Liverpool,i,0.0,Knee injury - Unknown return date,0.039602,0.00,0.000000
396,Bradley,Liverpool,i,0.0,Knee injury - Unknown return date,0.039602,0.00,0.000000
395,Gomez,Liverpool,i,0.0,Muscular injury - Unknown return date,0.039602,0.00,0.000000
390,Jaros,Liverpool,i,0.0,Knee injury - Unknown return date,0.039602,0.00,0.000000
...,...,...,...,...,...,...,...,...
144,Welbeck,Chelsea,d,75.0,Unspecified injury - 75% chance of playing,0.034547,0.75,0.025910
143,Henderson,Chelsea,d,75.0,Wrist injury - 75% chance of playing,0.034547,0.75,0.025910
409,C.Jones,Liverpool,d,75.0,Hip injury - 75% chance of playing,0.039602,0.75,0.029702
67,Soler,Bournemouth,d,75.0,Hamstring injury - 75% chance of playing,0.052418,0.75,0.039314


In [21]:
start_probs.groupby("Team")[
    "Start Prob"
].sum()

Team
Arsenal           11.0
Aston Villa       11.0
Bournemouth       11.0
Brentford         11.0
Brighton          11.0
Chelsea           11.0
Coventry City     11.0
Crystal Palace    11.0
Everton           11.0
Fulham            11.0
Hull City         11.0
Ipswich Town      11.0
Leeds             11.0
Liverpool         11.0
Man City          11.0
Man Utd           11.0
Newcastle         11.0
Nott'm Forest     11.0
Spurs             11.0
Sunderland        11.0
Name: Start Prob, dtype: float64

In [22]:
start_probs[
    start_probs["Availability Prob"] < 1
].groupby(
    ["Status", "Chance Play Next"],
    dropna=False,
).size()

Status  Chance Play Next
d       25.0                 1
        50.0                 2
        75.0                17
i       0.0                 51
s       0.0                  3
u       0.0                 38
dtype: int64

In [23]:
from src.historical_minutes import (
    fetch_historical_minute_data,
    build_minute_priors,
)

hist_minutes = fetch_historical_minute_data(
    DATA / "cache"
)

minute_priors = build_minute_priors(
    current_players,
    hist_minutes,
)

In [24]:
from src.minutes import build_expected_minutes

minutes = build_expected_minutes(
    start_probs,
    minute_priors,
)

In [25]:
minutes.groupby("Team")["Effective Mins"].sum().sort_values()

Team
Aston Villa        940.701635
Bournemouth        958.081901
Brighton           961.001468
Hull City          965.889725
Fulham             966.866941
Newcastle          969.810074
Everton            976.097459
Liverpool          980.278659
Leeds              985.572425
Spurs              994.338363
Ipswich Town       995.336473
Coventry City      995.774240
Arsenal            998.131738
Brentford         1002.072905
Nott'm Forest     1003.736736
Sunderland        1008.341588
Man City          1009.471147
Man Utd           1011.478554
Crystal Palace    1031.653757
Chelsea           1056.708850
Name: Effective Mins, dtype: float64

In [26]:
from src.attack_projection import build_attack_projection

attack = build_attack_projection(
    minutes,
    model["Attack_Priors"],
    model["Players"],
    model["Market_Odds"],
)

attack[attack["Team"] == "Arsenal"][
    [
        "Player",
        "Effective Mins",
        "Attack Prior Source",
        "Team xG",
        "xG Share Used",
        "GW1 xG",
        "GW1 xA",
    ]
].sort_values("GW1 xG", ascending=False)

,Player,Effective Mins,Attack Prior Source,Team xG,xG Share Used,GW1 xG,GW1 xA
23,Gyökeres,59.132959,Player prior,2.652936,0.293373,0.521716,0.077611
27,Tzolis,76.456713,Player prior,2.652936,0.107518,0.247219,0.207493
24,Havertz,22.533152,Player prior,2.652936,0.302587,0.205049,0.031207
7,Calafiori,72.522796,Player prior,2.652936,0.080205,0.174929,0.064804
12,Rice,82.733184,Player prior,2.652936,0.066392,0.165187,0.286839
14,Ødegaard,72.713746,Player prior,2.652936,0.073079,0.159806,0.259608
6,Lewis-Skelly,79.226649,Player prior,2.652936,0.066430,0.158277,0.141339
11,Saka,28.712871,Player prior,2.652936,0.156051,0.134750,0.124378
3,Gabriel,84.665231,Player prior,2.652936,0.051844,0.132006,0.089454
15,Madueke,51.948504,Player prior,2.652936,0.077843,0.121612,0.169579


In [27]:
check = (
    attack.groupby("Team")
    .agg(
        Player_xG=("GW1 xG", "sum"),
        Market_xG=("Team xG", "first"),
        Player_xA=("GW1 xA", "sum"),
    )
)

check["xG Error"] = check["Player_xG"] - check["Market_xG"]

check

,Player_xG,Market_xG,Player_xA,xG Error
Team,,,,
Arsenal,2.652936,2.652936,1.916777,0.000000e+00
Aston Villa,1.214704,1.214704,0.821666,0.000000e+00
Bournemouth,0.919178,0.919178,0.592955,1.110223e-16
Brentford,1.372034,1.372034,0.789051,0.000000e+00
Brighton,1.452548,1.452548,1.101336,0.000000e+00
Chelsea,1.529191,1.529191,0.867146,0.000000e+00
Coventry City,0.528659,0.528659,0.327266,-1.110223e-16
Crystal Palace,1.005421,1.005421,0.667609,0.000000e+00
Everton,1.304248,1.304248,0.804297,0.000000e+00


In [28]:
attack[attack["Attack Prior Source"] == "Position fallback"][
    ["Player", "Team", "FPL Pos", "Effective Mins"]
].sort_values("Effective Mins", ascending=False)

,Player,Team,FPL Pos,Effective Mins
322,Tzolakis,Hull City,GK,85.287223
536,Diomande,Nott'm Forest,DEF,81.077340
290,Palacios,Fulham,MID,76.975053
324,Stroud,Hull City,MID,76.665055
210,Yirenkyi,Coventry City,MID,75.941536
289,Gonzalo,Fulham,FWD,75.532608
509,Dedić,Newcastle,DEF,63.085447
326,Mendy,Hull City,DEF,62.783861
323,Hjertø-Dahl,Hull City,MID,59.738416
57,Suzuki,Aston Villa,GK,26.509192


In [29]:
import setuptools
from distutils.version import LooseVersion

import LanusStats as ls

fotmob = ls.FotMob()

print("LanusStats import worked")

LanusStats import worked


In [30]:
from src.fotmob_history import (
    fetch_external_history,
    match_external_priors,
)

REFRESH_FOTMOB = False

cache_file = DATA / "cache" / "fotmob_attack_history_2025_26.csv"

if REFRESH_FOTMOB and cache_file.exists():
    cache_file.unlink()

leagues = [
    "Premier League",
    "La Liga",
    "Bundesliga",
    "Serie A",
    "Ligue 1",
]

external_history = fetch_external_history(
    DATA / "cache",
    leagues,
)

print("External history:", external_history.shape)

External history: (2293, 10)


In [31]:
# Find players missing our existing workbook attacking prior
existing_priors = model["Attack_Priors"].merge(
    model["Players"][["Player ID", "Code"]],
    on="Player ID",
    how="left",
)

prior_by_code = existing_priors[
    ["Code", "Current xG Share Prior", "Current xA Share Prior"]
].drop_duplicates("Code")

fallback_players = current_players.merge(
    prior_by_code,
    on="Code",
    how="left",
)

fallback_players = fallback_players[
    fallback_players["Current xG Share Prior"].isna()
].copy()

external_priors = match_external_priors(
    fallback_players,
    external_history,
)

external_priors.sort_values("External Match Score")

,Player ID,Code,Player,External Player,External League,External Team,External xG Share,External xA Share,External Match Score
19,581,568420,Diomande,Yan Diomande,Bundesliga,RB Leipzig,0.105341,0.149852,0.98
2,597,433969,Suzuki,Yuito Suzuki,Bundesliga,Freiburg,0.079430,0.052953,0.98
4,594,538007,David,David Brooks,Premier League,AFC Bournemouth,0.093700,0.053312,0.98
10,580,463210,Charles,Charles Pickel,La Liga,Espanyol,0.016360,0.002045,0.98
17,579,445087,Araujo,Ronald Araujo,La Liga,Barcelona,0.016148,0.004614,1.00
16,598,173774,Elvedi,Nico Elvedi,Bundesliga,Borussia Mönchengladbach,0.011682,0.030374,1.00
15,592,572584,Ouattara,Abdoul Ouattara,Ligue 1,Strasbourg,0.046125,0.018450,1.00
14,591,474120,Enciso,Julio Enciso,Ligue 1,Strasbourg,0.140221,0.097786,1.00
13,571,216055,Florentino,Florentino,Premier League,Burnley,0.021538,0.018462,1.00
12,589,469266,Gourna-Douath,Lucas Gourna-Douath,Ligue 1,Le Havre,0.005208,0.010417,1.00


In [32]:
attack = build_attack_projection(
    minutes,
    model["Attack_Priors"],
    model["Players"],
    model["Market_Odds"],
    external_priors=external_priors,
)

In [33]:
print(attack.columns.tolist())

['Player ID', 'Code', 'Player', 'Team', 'FPL Pos', 'Status', 'Chance Play Next', 'News', 'FFScout', 'RotoWire', 'NMA', 'Lineup Votes', 'Lineup Consensus', 'Availability Prob', 'Lineup Evidence', 'Conditional Start Prob', 'Source Start Prob', 'Start Prob', 'Mins If Start', 'Expected Mins If Not Start', 'Start_Sample', 'Bench_Sample', 'Model Expected Mins', 'User Mins Override', 'Effective Mins', 'Current xG Share Prior', 'Current xA Share Prior', 'Uncertainty', 'History Quality', 'Penalty Contamination?', 'External xG Share', 'External xA Share', 'External League', 'External Team', 'Fallback_xG', 'Fallback_xA', 'xG Share Used', 'xA Share Used', 'Attack Prior Source', 'Opponent', 'Team xG', 'Opponent xG', 'H/A', 'Raw xG Weight', 'Raw xA Weight', 'Attack Reconciliation Factor', 'GW1 xG', 'GW1 xA']


In [34]:
attack[
    attack["Attack Prior Source"] != "Player prior"
][
    [
        "Player",
        "Team",
        "Effective Mins",
        "Attack Prior Source",
        "External League",
        "External Team",
        "xG Share Used",
        "xA Share Used",
    ]
].sort_values(
    "Effective Mins",
    ascending=False,
)

,Player,Team,Effective Mins,Attack Prior Source,External League,External Team,xG Share Used,xA Share Used
322,Tzolakis,Hull City,85.287223,Position fallback,NaN,NaN,0.000167,0.001379
536,Diomande,Nott'm Forest,81.077340,External prior,Bundesliga,RB Leipzig,0.105341,0.149852
290,Palacios,Fulham,76.975053,External prior,La Liga,Real Madrid,0.002567,0.001284
324,Stroud,Hull City,76.665055,Position fallback,NaN,NaN,0.107518,0.090241
210,Yirenkyi,Coventry City,75.941536,Position fallback,NaN,NaN,0.107518,0.090241
289,Gonzalo,Fulham,75.532608,External prior,La Liga,Real Madrid,0.069320,0.014121
509,Dedić,Newcastle,63.085447,Position fallback,NaN,NaN,0.040894,0.040510
326,Mendy,Hull City,62.783861,External prior,La Liga,Rayo Vallecano,0.030075,0.003759
323,Hjertø-Dahl,Hull City,59.738416,Position fallback,NaN,NaN,0.107518,0.090241
57,Suzuki,Aston Villa,26.509192,External prior,Bundesliga,Freiburg,0.079430,0.052953


In [35]:
attack["Needs Attack Review"] = (
    (attack["Attack Prior Source"] == "Position fallback")
    & (attack["Effective Mins"] >= 30)
    & (attack["FPL Pos"] != "GK")
)

attack[attack["Needs Attack Review"]][
    ["Player", "Team", "FPL Pos", "Effective Mins"]
]

,Player,Team,FPL Pos,Effective Mins
210,Yirenkyi,Coventry City,MID,75.941536
323,Hjertø-Dahl,Hull City,MID,59.738416
324,Stroud,Hull City,MID,76.665055
509,Dedić,Newcastle,DEF,63.085447


In [36]:
from src.appearance import (
    build_appearance_priors,
    build_appearance_probs,
)

appearance_priors = build_appearance_priors(
    current_players,
    hist_minutes,
)

appearance = build_appearance_probs(
    start_probs,
    appearance_priors,
)

In [37]:
appearance[
    appearance["Team"] == "Arsenal"
][
    [
        "Player",
        "Start Prob",
        "Appearance Prob",
        "P60",
        "Expected Appearance Pts",
    ]
].sort_values("Appearance Prob", ascending=False)

,Player,Start Prob,Appearance Prob,P60,Expected Appearance Pts
6,Lewis-Skelly,0.947801,0.969936,0.896745,1.866681
14,Ødegaard,0.947801,0.968745,0.754155,1.722900
10,Mosquera,0.947801,0.968106,0.867809,1.835915
27,Tzolis,0.947801,0.964685,0.864963,1.829648
12,Rice,0.947801,0.963590,0.931745,1.895334
3,Gabriel,0.947801,0.962107,0.937993,1.900100
7,Calafiori,0.947801,0.958779,0.837484,1.796262
9,White,0.947801,0.955015,0.815272,1.770287
0,Raya,0.947801,0.947963,0.947127,1.895090
23,Gyökeres,0.710761,0.888068,0.630170,1.518238


In [38]:
appearance[
    ["Appearance Prob", "P60"]
].describe()

,Appearance Prob,P60
count,599.000000,599.000000
mean,0.500878,0.347414
std,0.381160,0.406157
min,0.000000,0.000000
25%,0.172161,0.037768
50%,0.432842,0.046658
75%,0.959149,0.867661
max,0.989995,0.959627


In [39]:
from src.xpts import build_gw1_xpts

xpts = build_gw1_xpts(
    attack,
    appearance,
)

xpts[
    [
        "Player",
        "Team",
        "FPL Pos",
        "Effective Mins",
        "GW1 xG",
        "GW1 xA",
        "Team CS Prob",
        "GW1 Base xPts",
    ]
].sort_values(
    "GW1 Base xPts",
    ascending=False,
).head(40)

,Player,Team,FPL Pos,Effective Mins,GW1 xG,GW1 xA,Team CS Prob,GW1 Base xPts
3,Gabriel,Arsenal,DEF,84.665231,0.132006,0.089454,0.589395,5.080764
468,B.Fernandes,Man Utd,MID,83.315151,0.319097,0.347922,0.545824,5.040055
7,Calafiori,Arsenal,DEF,72.522796,0.174929,0.064804,0.589395,4.945042
469,Mbeumo,Man Utd,MID,78.483586,0.369739,0.176875,0.545824,4.783338
448,Haaland,Man City,FWD,81.726370,0.658363,0.073146,0.398847,4.741005
459,Dalot,Man Utd,DEF,81.497247,0.074338,0.091529,0.545824,4.481446
9,White,Arsenal,DEF,72.715170,0.083151,0.098446,0.589395,4.416636
10,Mosquera,Arsenal,DEF,77.792000,0.054607,0.086414,0.589395,4.390011
460,Maguire,Man Utd,DEF,80.284486,0.070011,0.046791,0.545824,4.386023
465,Shaw,Man Utd,DEF,81.854407,0.035402,0.064747,0.545824,4.242113


In [40]:
from src.defcon_calibration import (
    build_defcon_oof,
    fit_defender_calibrator,
)

dc_oof = build_defcon_oof(
    hist_minutes,
    shrink_k=4,
)

dc_calibrator = fit_defender_calibrator(
    dc_oof,
)

print("DefCon calibrator:", dc_calibrator)

DefCon calibrator: {'a': np.float64(-0.8078270552541189), 'b': np.float64(0.8184037500133496), 'raw_brier': np.float64(0.21648580726050654), 'calibrated_brier': np.float64(0.18772732132091677), 'n': 2130}


In [41]:
from src.defensive_points import (
    build_defensive_priors,
    build_defensive_xpts,
)

defensive_priors = build_defensive_priors(
    current_players,
    hist_minutes,
)

defensive_xpts = build_defensive_xpts(
    start_probs,
    defensive_priors,
    defcon_calibrator=dc_calibrator,
)

In [42]:
xpts = xpts.merge(
    defensive_xpts[
        [
            "Player ID",
            "P Defensive Return",
            "xPts DefCon",
            "xPts Saves",
        ]
    ],
    on="Player ID",
    how="left",
)

xpts["GW1 xPts"] = (
    xpts["GW1 Base xPts"]
    + xpts["xPts DefCon"]
    + xpts["xPts Saves"]
)

In [43]:
xpts[
    [
        "Player",
        "Team",
        "FPL Pos",
        "Effective Mins",
        "GW1 Base xPts",
        "xPts DefCon",
        "xPts Saves",
        "GW1 xPts",
    ]
].sort_values(
    "GW1 xPts",
    ascending=False,
).head(40)

,Player,Team,FPL Pos,Effective Mins,GW1 Base xPts,xPts DefCon,xPts Saves,GW1 xPts
3,Gabriel,Arsenal,DEF,84.665231,5.080764,0.482000,0.000000,5.562764
468,B.Fernandes,Man Utd,MID,83.315151,5.040055,0.282745,0.000000,5.322800
7,Calafiori,Arsenal,DEF,72.522796,4.945042,0.297554,0.000000,5.242596
460,Maguire,Man Utd,DEF,80.284486,4.386023,0.469096,0.000000,4.855118
12,Rice,Arsenal,MID,82.733184,4.130955,0.700678,0.000000,4.831633
469,Mbeumo,Man Utd,MID,78.483586,4.783338,0.037657,0.000000,4.820996
10,Mosquera,Arsenal,DEF,77.792000,4.390011,0.424238,0.000000,4.814250
449,Anderson,Man City,MID,84.337442,3.575280,1.226831,0.000000,4.802111
448,Haaland,Man City,FWD,81.726370,4.741005,0.002799,0.000000,4.743804
459,Dalot,Man Utd,DEF,81.497247,4.481446,0.198760,0.000000,4.680207


In [44]:
from src.bonus_points import (
    build_bonus_priors,
    build_bonus_xpts,
)

bonus_priors = build_bonus_priors(
    current_players,
    hist_minutes,
)

bonus_xpts = build_bonus_xpts(
    start_probs,
    bonus_priors,
)

In [45]:
xpts = xpts.merge(
    bonus_xpts[
        ["Player ID", "xPts Bonus"]
    ],
    on="Player ID",
    how="left",
)

xpts["GW1 xPts Model"] = (
    xpts["GW1 xPts"]
)

In [46]:
xpts[
    [
        "Player",
        "Team",
        "FPL Pos",
        "Effective Mins",
        "GW1 xPts",
        "GW1 xPts Model",
    ]
].sort_values(
    "GW1 xPts Model",
    ascending=False,
).head(40)

,Player,Team,FPL Pos,Effective Mins,GW1 xPts,GW1 xPts Model
3,Gabriel,Arsenal,DEF,84.665231,5.562764,5.562764
468,B.Fernandes,Man Utd,MID,83.315151,5.322800,5.322800
7,Calafiori,Arsenal,DEF,72.522796,5.242596,5.242596
460,Maguire,Man Utd,DEF,80.284486,4.855118,4.855118
12,Rice,Arsenal,MID,82.733184,4.831633,4.831633
469,Mbeumo,Man Utd,MID,78.483586,4.820996,4.820996
10,Mosquera,Arsenal,DEF,77.792000,4.814250,4.814250
449,Anderson,Man City,MID,84.337442,4.802111,4.802111
448,Haaland,Man City,FWD,81.726370,4.743804,4.743804
459,Dalot,Man Utd,DEF,81.497247,4.680207,4.680207


In [47]:
price_map = current_players[
    ["Player ID", "Current £m"]
].drop_duplicates("Player ID")

xpts = xpts.merge(
    price_map,
    on="Player ID",
    how="left",
)

xpts[
    ["Player", "Team", "FPL Pos", "Current £m", "GW1 xPts Model"]
].sort_values(
    "GW1 xPts Model",
    ascending=False,
).head(30)

,Player,Team,FPL Pos,Current £m,GW1 xPts Model
3,Gabriel,Arsenal,DEF,8.0,5.562764
468,B.Fernandes,Man Utd,MID,12.0,5.322800
7,Calafiori,Arsenal,DEF,5.5,5.242596
460,Maguire,Man Utd,DEF,5.0,4.855118
12,Rice,Arsenal,MID,7.5,4.831633
469,Mbeumo,Man Utd,MID,8.0,4.820996
10,Mosquera,Arsenal,DEF,5.5,4.814250
449,Anderson,Man City,MID,6.5,4.802111
448,Haaland,Man City,FWD,15.5,4.743804
459,Dalot,Man Utd,DEF,5.0,4.680207


In [48]:
from src.optimizer import optimise_gw1_team

data_team = optimise_gw1_team(
    xpts,
    budget=100.0,
)

In [49]:
data_team[
    [
        "Player",
        "Team",
        "FPL Pos",
        "Current £m",
        "GW1 xPts Model",
        "Starter",
        "Captain",
    ]
].sort_values(
    ["Starter", "FPL Pos", "GW1 xPts Model"],
    ascending=[False, True, False],
)

,Player,Team,FPL Pos,Current £m,GW1 xPts Model,Starter,Captain
3,Gabriel,Arsenal,DEF,8.0,5.562764,True,True
5,Calafiori,Arsenal,DEF,5.5,5.242596,True,False
386,Maguire,Man Utd,DEF,5.0,4.855118,True,False
455,Diomande,Nott'm Forest,DEF,5.5,4.515699,True,False
488,Ballard,Sunderland,DEF,5.0,4.169356,True,False
377,Haaland,Man City,FWD,15.5,4.743804,True,False
383,Lammens,Man Utd,GK,5.0,4.327426,True,False
393,B.Fernandes,Man Utd,MID,12.0,5.322800,True,False
10,Rice,Arsenal,MID,7.5,4.831633,True,False
378,Anderson,Man City,MID,6.5,4.802111,True,False


In [50]:
print(
    "Cost:",
    data_team["Current £m"].sum()
)

print(
    "Starting XI xPts:",
    data_team.loc[
        data_team["Starter"],
        "GW1 xPts Model"
    ].sum()
)

captain = data_team[
    data_team["Captain"]
].iloc[0]

print(
    "Captain:",
    captain["Player"],
    captain["GW1 xPts Model"],
)

print(
    "Total incl captain:",
    data_team.loc[
        data_team["Starter"],
        "GW1 xPts Model"
    ].sum()
    + captain["GW1 xPts Model"]
)

Cost: 100.0
Starting XI xPts: 52.54129568019514
Captain: Gabriel 5.562764415029395
Total incl captain: 58.10406009522454


In [51]:
from src.defcon_validation import validate_defcon

dc_validation = validate_defcon(
    hist_minutes,
    train_end_gw=25,
    shrink_k=8,
)

print(
    "Model Brier:",
    dc_validation["brier"],
)

print(
    "Position-only baseline:",
    dc_validation["baseline_brier"],
)

Model Brier: 0.14453578532172381
Position-only baseline: 0.1584907361628775


In [52]:
print(dc_calibrator)

{'a': np.float64(-0.8078270552541189), 'b': np.float64(0.8184037500133496), 'raw_brier': np.float64(0.21648580726050654), 'calibrated_brier': np.float64(0.18772732132091677), 'n': 2130}


In [53]:
for sheet in [
    "Fixtures",
    "Market_Odds",
]:
    print("\n" + "=" * 80)
    print(sheet)
    print("Shape:", model[sheet].shape)
    print("Columns:", model[sheet].columns.tolist())

    display(
        model[sheet].head(10)
    )


Fixtures
Shape: (760, 14)
Columns: ['GW', 'Fixture ID', 'Kickoff UTC', 'Team', 'Opponent', 'H/A', 'Official FDR', 'Market Team xG', 'Market Opp xG', 'CS Prob', 'Attack Fixture Adj', 'Def Fixture Adj', 'Notes', 'Source URL']


,GW,Fixture ID,Kickoff UTC,Team,Opponent,H/A,Official FDR,Market Team xG,Market Opp xG,CS Prob,Attack Fixture Adj,Def Fixture Adj,Notes,Source URL
0,1,1,46255.791667,Coventry City,Arsenal,A,5,NaN,NaN,NaN,1,1,NaN,https://fantasy.premierleague.com/
1,1,1,46255.791667,Arsenal,Coventry City,H,2,NaN,NaN,NaN,1,1,NaN,https://fantasy.premierleague.com/
2,1,2,46256.687500,Spurs,Brentford,A,3,NaN,NaN,NaN,1,1,NaN,https://fantasy.premierleague.com/
3,1,2,46256.687500,Brentford,Spurs,H,3,NaN,NaN,NaN,1,1,NaN,https://fantasy.premierleague.com/
4,1,3,46256.583333,Crystal Palace,Everton,A,3,NaN,NaN,NaN,1,1,NaN,https://fantasy.premierleague.com/
5,1,3,46256.583333,Everton,Crystal Palace,H,3,NaN,NaN,NaN,1,1,NaN,https://fantasy.premierleague.com/
6,1,4,46256.479167,Man Utd,Hull City,A,2,NaN,NaN,NaN,1,1,NaN,https://fantasy.premierleague.com/
7,1,4,46256.479167,Hull City,Man Utd,H,4,NaN,NaN,NaN,1,1,NaN,https://fantasy.premierleague.com/
8,1,5,46256.583333,Sunderland,Ipswich Town,A,2,NaN,NaN,NaN,1,1,NaN,https://fantasy.premierleague.com/
9,1,5,46256.583333,Ipswich Town,Sunderland,H,2,NaN,NaN,NaN,1,1,NaN,https://fantasy.premierleague.com/



Market_Odds
Shape: (10, 25)
Columns: ['GW', 'Fixture Seq', 'Home', 'Away', 'H Odds', 'D Odds', 'A Odds', 'H Decimal', 'D Decimal', 'A Decimal', 'Raw H Prob', 'Raw D Prob', 'Raw A Prob', 'Market Sum', 'Fair H Prob', 'Fair D Prob', 'Fair A Prob', 'Market Home xG', 'Market Away xG', 'Market Total xG', 'Home CS Prob', 'Away CS Prob', 'Poisson Fit Error', 'Method', 'Source URL']


,GW,Fixture Seq,Home,Away,H Odds,D Odds,A Odds,H Decimal,D Decimal,A Decimal,...,Fair D Prob,Fair A Prob,Market Home xG,Market Away xG,Market Total xG,Home CS Prob,Away CS Prob,Poisson Fit Error,Method,Source URL
0,1,1,Arsenal,Coventry City,1/5,7/1,18/1,1.200,8.00,19.0,...,0.123644,0.052061,2.652936,0.528659,3.181595,0.589395,0.070444,2.928673e-08,Oddschecker best-price consensus; normalized t...,https://www.oddschecker.com/football/english/p...
1,1,2,Hull City,Man Utd,17/2,4/1,2/5,9.500,5.00,1.4,...,0.196165,0.700590,0.605459,1.984312,2.589771,0.137475,0.545824,1.106112e-10,Oddschecker best-price consensus; normalized t...,https://www.oddschecker.com/football/english/p...
2,1,3,Ipswich Town,Sunderland,9/5,12/5,17/10,2.800,3.40,2.7,...,0.287890,0.362529,1.108826,1.134760,2.243586,0.321499,0.329946,4.795608e-13,Oddschecker best-price consensus; normalized t...,https://www.oddschecker.com/football/english/p...
3,1,4,Everton,Crystal Palace,5/4,5/2,12/5,2.250,3.50,3.4,...,0.278943,0.287147,1.304248,1.005421,2.309669,0.365890,0.271377,9.934099e-10,Oddschecker best-price consensus; normalized t...,https://www.oddschecker.com/football/english/p...
4,1,5,Nott'm Forest,Leeds,11/8,5/2,11/5,2.375,3.50,3.2,...,0.280314,0.306593,1.266981,1.050251,2.317232,0.349850,0.281681,6.544100e-09,Oddschecker best-price consensus; normalized t...,https://www.oddschecker.com/football/english/p...
5,1,6,Brentford,Spurs,6/4,11/4,9/5,2.500,3.75,2.8,...,0.260465,0.348837,1.372034,1.281860,2.653894,0.277521,0.253591,1.713629e-13,Oddschecker best-price consensus; normalized t...,https://www.oddschecker.com/football/english/p...
6,1,7,Man City,Bournemouth,1/2,4/1,11/2,1.500,5.00,6.5,...,0.195980,0.150754,2.138793,0.919178,3.057971,0.398847,0.117797,3.627099e-13,Oddschecker best-price consensus; normalized t...,https://www.oddschecker.com/football/english/p...
7,1,8,Brighton,Aston Villa,13/10,14/5,21/10,2.300,3.80,3.1,...,0.257866,0.316094,1.452548,1.214704,2.667252,0.296798,0.233973,6.394394e-10,Oddschecker best-price consensus; normalized t...,https://www.oddschecker.com/football/english/p...
8,1,9,Newcastle,Liverpool,27/10,29/10,1/1,3.700,3.90,2.0,...,0.249747,0.487006,1.100692,1.590353,2.691045,0.203854,0.332641,7.013478e-08,Oddschecker best-price consensus; normalized t...,https://www.oddschecker.com/football/english/p...
9,1,10,Fulham,Chelsea,14/5,14/5,1/1,3.800,3.80,2.0,...,0.256410,0.487179,1.034539,1.529191,2.563730,0.216711,0.355390,2.258289e-09,Oddschecker best-price consensus; normalized t...,https://www.oddschecker.com/football/english/p...


In [54]:
print("FIXTURES")
for col in model["Fixtures"].columns:
    print(
        col,
        "->",
        model["Fixtures"][col]
        .dropna()
        .astype(str)
        .unique()[:15]
    )

print("\nMARKET ODDS")
for col in model["Market_Odds"].columns:
    print(
        col,
        "->",
        model["Market_Odds"][col]
        .dropna()
        .astype(str)
        .unique()[:15]
    )

FIXTURES
GW -> <ArrowStringArray>
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14',
 '15']
Length: 15, dtype: str
Fixture ID -> <ArrowStringArray>
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14',
 '15']
Length: 15, dtype: str
Kickoff UTC -> <ArrowStringArray>
['46255.791666666664',         '46256.6875', '46256.583333333336',
 '46256.479166666664', '46257.541666666664', '46257.645833333336',
 '46258.791666666664', '46262.791666666664', '46263.583333333336',
 '46263.479166666664',         '46263.6875', '46264.541666666664',
 '46264.645833333336', '46265.791666666664', '46269.791666666664']
Length: 15, dtype: str
Team -> <ArrowStringArray>
[ 'Coventry City',        'Arsenal',          'Spurs',      'Brentford',
 'Crystal Palace',        'Everton',        'Man Utd',      'Hull City',
     'Sunderland',   'Ipswich Town',          'Leeds',  'Nott'm Forest',
    'Aston Villa',       'Brighton',    'Bournemouth']
Length: 15, dtype: str
Oppon

In [55]:
from src.fixture_projection import (
    fit_fixture_model,
    project_team_fixtures,
)

fixture_horizon, fixture_fit = (
    project_team_fixtures(
        model["Fixtures"],
        team_hist,
        model["Market_Odds"],
        max_gw=6,
        ridge_lambda=2.0,
    )
)

In [56]:
from src.fixture_projection import (
    fit_fixture_model,
    project_team_fixtures,
)

fixture_horizon, fixture_fit = (
    project_team_fixtures(
        model["Fixtures"],
        team_hist,
        model["Market_Odds"],
        max_gw=6,
        ridge_lambda=2.0,
    )
)

In [57]:
fixture_fit[
    "fit_table"
].sort_values(
    "Abs Error",
    ascending=False,
)

,GW,Fixture Seq,Team,Opponent,H/A,Market Team xG,Model xG,Error,Abs Error
0,1,1,Arsenal,Coventry City,H,2.652936,1.644022,-1.008914,1.008914
11,1,2,Man Utd,Hull City,A,1.984312,2.363160,0.378848,0.378848
13,1,4,Crystal Palace,Everton,A,1.005421,1.379731,0.374310,0.374310
15,1,6,Spurs,Brentford,A,1.281860,0.946291,-0.335569,0.335569
6,1,7,Man City,Bournemouth,H,2.138793,1.819284,-0.319508,0.319508
18,1,9,Liverpool,Newcastle,A,1.590353,1.317823,-0.272530,0.272530
14,1,5,Leeds,Nott'm Forest,A,1.050251,1.310969,0.260719,0.260719
12,1,3,Sunderland,Ipswich Town,A,1.134760,0.875950,-0.258811,0.258811
16,1,7,Bournemouth,Man City,A,0.919178,1.151744,0.232566,0.232566
1,1,2,Hull City,Man Utd,H,0.605459,0.831464,0.226006,0.226006


In [58]:
fixture_horizon[
    fixture_horizon["Team"].isin(
        [
            "Arsenal",
            "Man City",
            "Man Utd",
            "Liverpool",
        ]
    )
][
    [
        "GW",
        "Team",
        "Opponent",
        "H/A",
        "Team xG",
        "Opp xG",
        "CS Prob",
        "xG Source",
    ]
]

,GW,Team,Opponent,H/A,Team xG,Opp xG,CS Prob,xG Source
1,1,Arsenal,Coventry City,H,2.652936,0.528659,0.589395,Market
6,1,Man Utd,Hull City,A,1.984312,0.605459,0.545824,Market
15,1,Man City,Bournemouth,H,2.138793,0.919178,0.398847,Market
16,1,Liverpool,Newcastle,A,1.590353,1.100692,0.332641,Market
20,2,Man City,Crystal Palace,A,1.534569,1.183593,0.306177,Strength model
27,2,Liverpool,Nott'm Forest,H,1.593330,0.940821,0.390307,Strength model
35,2,Man Utd,Ipswich Town,H,1.523901,0.988946,0.371968,Strength model
38,2,Arsenal,Aston Villa,A,1.469906,0.648998,0.522569,Strength model
40,3,Liverpool,Ipswich Town,A,1.323932,1.049222,0.350210,Strength model
51,3,Man City,Coventry City,H,1.780037,0.985974,0.373076,Strength model


In [59]:
fixture_horizon.groupby(
    "GW"
).agg(
    Teams=("Team", "size"),
    Mean_xG=("Team xG", "mean"),
    Min_xG=("Team xG", "min"),
    Max_xG=("Team xG", "max"),
    Mean_CS=("CS Prob", "mean"),
)

,Teams,Mean_xG,Min_xG,Max_xG,Mean_CS
GW,,,,,
1,20,1.313787,0.528659,2.652936,0.297525
2,20,1.271216,0.648998,2.094444,0.294844
3,20,1.260978,0.799897,1.793608,0.294473
4,20,1.284044,0.534182,2.611161,0.295540
5,20,1.272433,0.755314,2.273639,0.295228
6,20,1.264447,0.664298,1.742839,0.294198


In [60]:
fixture_horizon[
    fixture_horizon["GW"] == 1
][
    [
        "Team",
        "Opponent",
        "H/A",
        "Team xG",
        "Opp xG",
        "CS Prob",
        "xG Source",
    ]
].sort_values("Team")

,Team,Opponent,H/A,Team xG,Opp xG,CS Prob,xG Source
1,Arsenal,Coventry City,H,2.652936,0.528659,0.589395,Market
12,Aston Villa,Brighton,A,1.214704,1.452548,0.233973,Market
14,Bournemouth,Man City,A,0.919178,2.138793,0.117797,Market
3,Brentford,Spurs,H,1.372034,1.281860,0.277521,Market
13,Brighton,Aston Villa,H,1.452548,1.214704,0.296798,Market
18,Chelsea,Fulham,A,1.529191,1.034539,0.355390,Market
0,Coventry City,Arsenal,A,0.528659,2.652936,0.070444,Market
4,Crystal Palace,Everton,A,1.005421,1.304248,0.271377,Market
5,Everton,Crystal Palace,H,1.304248,1.005421,0.365890,Market
19,Fulham,Chelsea,H,1.034539,1.529191,0.216711,Market


In [61]:
fixture_horizon[
    fixture_horizon["GW"] == 1
]["xG Source"].value_counts()

xG Source
Market    20
Name: count, dtype: int64

In [62]:
print("RMSE:", fixture_fit["rmse"])
print("MAE:", fixture_fit["mae"])

for k, v in fixture_fit["coefficients"].items():
    print(f"{k:24s}: {v:.4f}")

RMSE: 0.3142565009990046
MAE: 0.23735524984069709
Intercept               : -0.1507
Home                    : 0.0833
Log Attack              : 0.9082
Log Opp Def Weakness    : 0.9636
Promoted Team           : -0.2942
Promoted Opponent       : 0.2813


In [63]:
print(model["Attack_Priors"].columns.tolist())

['Player ID', 'Player', 'Team', 'FPL Pos', 'Hist League', 'Hist Team', 'Hist Mins', 'Hist xG', 'Hist xA', 'Hist xG90', 'Hist xA90', 'Hist Team xG', 'Hist Team xG/Match', 'Hist xG Share', 'Hist xA Share', 'Base Min Weight', 'League Reliability', 'Team Changed?', 'Transport Reliability', 'New Manager Summer?', 'Manager Reliability', 'Effective Weight', 'Pos xG Share Prior', 'Pos xA Share Prior', 'xG Share Prior', 'xA Share Prior', 'Role xG Share Adj', 'Role xA Share Adj', 'Current xG Share Prior', 'Current xA Share Prior', 'Uncertainty', 'History Quality', 'Pens Order', 'Direct FK Order', 'Corners/IFK Order', 'Manual Review?', 'History Source', 'Source URL', 'Notes', 'Hist Pens Order', 'Hist Pens Missed', 'Penalty Contamination?', 'npxG Status']


In [64]:
print(hist_minutes.columns.tolist())

['id', 'first_name', 'second_name', 'web_name', 'status', 'news', 'news_added', 'now_cost', 'now_cost_rank', 'now_cost_rank_type', 'selected_by_percent', 'selected_rank', 'selected_rank_type', 'form', 'form_rank', 'form_rank_type', 'event_points', 'cost_change_event', 'cost_change_event_fall', 'cost_change_start', 'cost_change_start_fall', 'transfers_in_event', 'transfers_out_event', 'value_form', 'value_season', 'ep_next', 'ep_this', 'points_per_game', 'points_per_game_rank', 'points_per_game_rank_type', 'chance_of_playing_next_round', 'chance_of_playing_this_round', 'influence_rank', 'influence_rank_type', 'creativity_rank', 'creativity_rank_type', 'threat_rank', 'threat_rank_type', 'ict_index_rank', 'ict_index_rank_type', 'corners_and_indirect_freekicks_order', 'direct_freekicks_order', 'penalties_order', 'set_piece_threat', 'corners_and_indirect_freekicks_text', 'direct_freekicks_text', 'penalties_text', 'expected_goals_per_90', 'expected_assists_per_90', 'expected_goal_involvement

In [65]:
priors = model["Attack_Priors"]

print("Penalty Contamination?")
print(
    priors["Penalty Contamination?"]
    .value_counts(dropna=False)
)

print("\nHist Pens Order")
print(
    priors["Hist Pens Order"]
    .value_counts(dropna=False)
    .head(20)
)

print("\nnpxG Status")
print(
    priors["npxG Status"]
    .value_counts(dropna=False)
)

Penalty Contamination?
Penalty Contamination?
0.0    430
NaN    110
1.0     27
Name: count, dtype: int64

Hist Pens Order
Hist Pens Order
NaN    518
2.0     15
3.0     14
1.0      9
4.0      8
5.0      3
Name: count, dtype: int64

npxG Status
npxG Status
Total xG retained; low penalty contamination         430
No comparable 25/26 PL penalty history               110
Total xG retained; penalty channel not decomposed     27
Name: count, dtype: int64


In [66]:
priors[
    priors["Penalty Contamination?"].notna()
][
    [
        "Player",
        "Hist Pens Order",
        "Hist Pens Missed",
        "Penalty Contamination?",
        "npxG Status",
    ]
].head(30)

,Player,Hist Pens Order,Hist Pens Missed,Penalty Contamination?,npxG Status
0,Raya,NaN,0.0,0.0,Total xG retained; low penalty contamination
1,Arrizabalaga,NaN,0.0,0.0,Total xG retained; low penalty contamination
2,Meslier,NaN,0.0,0.0,Total xG retained; low penalty contamination
3,Gabriel,NaN,0.0,0.0,Total xG retained; low penalty contamination
4,J.Timber,NaN,0.0,0.0,Total xG retained; low penalty contamination
5,Saliba,NaN,0.0,0.0,Total xG retained; low penalty contamination
6,Lewis-Skelly,NaN,0.0,0.0,Total xG retained; low penalty contamination
7,Calafiori,NaN,0.0,0.0,Total xG retained; low penalty contamination
8,Hincapie,NaN,0.0,0.0,Total xG retained; low penalty contamination
9,White,NaN,0.0,0.0,Total xG retained; low penalty contamination


In [67]:
from src.finishing_validation import (
    validate_finishing,
)

results = []

for k in [
    1,
    2,
    5,
    10,
    15,
    20,
    30,
    50,
    100,
]:

    v = validate_finishing(
        hist_minutes,
        model["Attack_Priors"],
        shrink_k=k,
    )

    results.append({
        "K": k,
        "Baseline Deviance":
            v["baseline_deviance"],
        "Finishing Deviance":
            v["finishing_deviance"],
        "Improvement %":
            100 * (
                v["baseline_deviance"]
                - v["finishing_deviance"]
            )
            / v["baseline_deviance"],
    })

finishing_k = pd.DataFrame(results)

finishing_k.sort_values(
    "Finishing Deviance"
)

,K,Baseline Deviance,Finishing Deviance,Improvement %
3,10,0.331072,0.328863,0.667409
4,15,0.331072,0.328900,0.656269
5,20,0.331072,0.329112,0.592243
6,30,0.331072,0.329502,0.474386
7,50,0.331072,0.329979,0.330100
8,100,0.331072,0.330460,0.184859
2,5,0.331072,0.330469,0.182335
1,2,0.331072,0.339619,-2.581451
0,1,0.331072,0.355269,-7.308480


In [68]:
best_k = (
    finishing_k
    .sort_values("Finishing Deviance")
    .iloc[0]["K"]
)

finishing_validation = (
    validate_finishing(
        hist_minutes,
        model["Attack_Priors"],
        shrink_k=best_k,
    )
)

print("Best K:", best_k)

print(
    "Raw xG deviance:",
    finishing_validation[
        "baseline_deviance"
    ],
)

print(
    "Finishing-adjusted:",
    finishing_validation[
        "finishing_deviance"
    ],
)

Best K: 10.0
Raw xG deviance: 0.33107233751180093
Finishing-adjusted: 0.3288627319353327


In [69]:
finishing_validation[
    "oof"
].groupby(
    ["Train Through", "Test GWs"]
).agg(
    N=("player_id", "size"),
    Baseline=(
        "Baseline Deviance",
        "mean",
    ),
    Finishing=(
        "Finishing Deviance",
        "mean",
    ),
)

,,N,Baseline,Finishing
Train Through,Test GWs,,,
12,13-18,379,0.294871,0.285572
18,19-25,379,0.299818,0.302937
25,26-38,379,0.398527,0.398079


In [70]:
print([
    c for c in xpts.columns
    if "xPts" in c or "Bonus" in c
])

wanted = [
    c for c in [
        "Player",
        "GW1 Base xPts",
        "xPts DefCon",
        "xPts Saves",
        "GW1 xPts",
        "xPts Bonus",
        "GW1 xPts Final",
        "GW1 xPts Model",
    ]
    if c in xpts.columns
]

display(
    xpts[
        xpts["Player"].isin(
            ["Gabriel", "Haaland", "B.Fernandes"]
        )
    ][wanted]
)

['xPts Appearance', 'xPts Goals', 'xPts Assists', 'xPts Clean Sheet', 'xPts Goals Conceded', 'Base xPts', 'GW1 Base xPts', 'xPts DefCon', 'xPts Saves', 'GW1 xPts', 'xPts Bonus', 'GW1 xPts Model']


,Player,GW1 Base xPts,xPts DefCon,xPts Saves,GW1 xPts,xPts Bonus,GW1 xPts Model
3,Gabriel,5.080764,0.482000,0.0,5.562764,0.835603,5.562764
448,Haaland,4.741005,0.002799,0.0,4.743804,1.095780,4.743804
468,B.Fernandes,5.040055,0.282745,0.0,5.322800,0.978237,5.322800


In [71]:
city = attack[
    attack["Team"] == "Man City"
].copy()

city["Team xG Share Final"] = (
    city["GW1 xG"]
    / city["GW1 xG"].sum()
)

city[
    [
        "Player",
        "Effective Mins",
        "Start Prob",
        "Conditional Start Prob",
        "xG Share Used",
        "GW1 xG",
        "GW1 xA",
        "Team xG Share Final",
    ]
].sort_values(
    "GW1 xG",
    ascending=False,
)

,Player,Effective Mins,Start Prob,Conditional Start Prob,xG Share Used,GW1 xG,GW1 xA,Team xG Share Final
448,Haaland,81.726370,0.955595,0.955595,0.377127,0.658363,0.073146,0.307820
434,Semenyo,82.149283,0.955595,0.955595,0.147405,0.258661,0.105805,0.120938
435,Foden,79.689774,0.955595,0.955595,0.126408,0.215175,0.173441,0.100606
443,Kovačić,72.981177,0.955595,0.955595,0.101734,0.158595,0.151630,0.074152
449,Anderson,84.337442,0.955595,0.955595,0.082854,0.149261,0.180208,0.069788
424,O'Reilly,66.507362,0.744405,0.744405,0.086497,0.122881,0.064735,0.057453
428,Gvardiol,80.020600,0.955595,0.955595,0.060550,0.103498,0.054925,0.048391
436,Cherki,32.333613,0.282720,0.282720,0.112035,0.077379,0.121475,0.036179
438,Marmoush,9.928282,0.050642,0.050642,0.263260,0.055831,0.009415,0.026104
430,Khusanov,81.114348,0.955595,0.955595,0.030884,0.053512,0.070185,0.025020


In [72]:
print(
    "City team xG:",
    city["GW1 xG"].sum()
)

print(
    "Haaland xG:",
    city.loc[
        city["Player"] == "Haaland",
        "GW1 xG",
    ].sum()
)

print(
    "xG allocated to <50% starters:",
    city.loc[
        city["Start Prob"] < 0.50,
        "GW1 xG",
    ].sum()
)

print(
    "Share allocated to <50% starters:",
    city.loc[
        city["Start Prob"] < 0.50,
        "GW1 xG",
    ].sum()
    / city["GW1 xG"].sum()
)

City team xG: 2.1387927682342065
Haaland xG: 0.6583625945720766
xG allocated to <50% starters: 0.37623735519041157
Share allocated to <50% starters: 0.1759110844109662


In [73]:
# Current RotoWire role attached to our attack projection
role_map = (
    rw_match[
        rw_match["Player ID"].notna()
    ][
        [
            "Player ID",
            "Position Raw",
        ]
    ]
    .drop_duplicates("Player ID")
)

role_audit = attack.merge(
    role_map,
    on="Player ID",
    how="left",
)

# Only use players with meaningful expected minutes
role_sample = role_audit[
    role_audit["Effective Mins"] >= 40
].copy()

role_summary = (
    role_sample
    .groupby("Position Raw")
    .agg(
        N=("Player ID", "size"),
        Median_xG_Share=("xG Share Used", "median"),
        Mean_xG_Share=("xG Share Used", "mean"),
        Median_xA_Share=("xA Share Used", "median"),
        Mean_xA_Share=("xA Share Used", "mean"),
    )
    .sort_values("Median_xG_Share")
)

display(role_summary)

,N,Median_xG_Share,Mean_xG_Share,Median_xA_Share,Mean_xA_Share
Position Raw,,,,,
GK,19,0.000054,0.000080,0.001379,0.001446
DL,16,0.035794,0.041878,0.056295,0.057737
DR,15,0.038024,0.036871,0.048521,0.047069
DC,41,0.041142,0.043546,0.033275,0.037557
ML,4,0.043033,0.043199,0.042299,0.049958
MR,4,0.043284,0.052890,0.051449,0.065354
MC,16,0.071005,0.078199,0.090241,0.091150
DMC,25,0.076894,0.081771,0.074317,0.082334
AMR,10,0.107518,0.111349,0.090559,0.095186


In [74]:
display(
    role_sample[
        role_sample["Position Raw"].isin(
            ["DMC", "MC", "DC", "DL", "DR", "AMC", "AML", "AMR", "FW"]
        )
    ][
        [
            "Player",
            "Team",
            "Position Raw",
            "Effective Mins",
            "xG Share Used",
            "xA Share Used",
        ]
    ]
    .sort_values(
        ["Position Raw", "xG Share Used"],
        ascending=[True, False],
    )
)

,Player,Team,Position Raw,Effective Mins,xG Share Used,xA Share Used
504,Wissa,Newcastle,AMC,71.803690,0.322405,0.046405
223,Sarr,Crystal Palace,AMC,80.223438,0.223426,0.054121
523,Gibbs-White,Nott'm Forest,AMC,81.659034,0.210726,0.076309
159,Palmer,Chelsea,AMC,74.452830,0.208816,0.075533
468,B.Fernandes,Man Utd,AMC,83.315151,0.165469,0.180416
...,...,...,...,...,...,...
373,Ampadu,Leeds,MC,84.826239,0.063941,0.055611
101,Janelt,Brentford,MC,78.227562,0.063281,0.089970
224,Wharton,Crystal Palace,MC,78.797400,0.060626,0.143112
164,Caicedo,Chelsea,MC,79.898172,0.053924,0.059439


In [75]:
import inspect

src = inspect.getsource(
    build_attack_projection
).splitlines()

for start, end in [
    (70, 120),
    (120, 160),
    (160, 185),
]:
    print("\n" + "=" * 90)
    print(f"LINES {start + 1}-{end}")

    for i in range(start, min(end, len(src))):
        print(f"{i + 1:03d}: {src[i]}")


LINES 71-120
071:             [
072:                 "Code",
073:                 "External xG Share",
074:                 "External xA Share",
075:                 "External League",
076:                 "External Team",
077:             ]
078:         ].drop_duplicates("Code")
079: 
080:         df = df.merge(
081:             external,
082:             on="Code",
083:             how="left",
084:         )
085: 
086:     else:
087:         df["External xG Share"] = np.nan
088:         df["External xA Share"] = np.nan
089:         df["External League"] = pd.NA
090:         df["External Team"] = pd.NA
091: 
092:     # --------------------------------
093:     # 5. Position fallback
094:     # --------------------------------
095: 
096:     df = df.merge(
097:         fallback,
098:         on="FPL Pos",
099:         how="left",
100:     )
101: 
102:     # --------------------------------
103:     # 6. Choose attacking prior source
104:     # --------------------------------
105: 
10

In [76]:
for i in range(125, 155):
    print(f"{i + 1:03d}: {src[i]}")

126:         ],
127:         default="Position fallback",
128:     )
129: 
130:     # ------------------------------
131:     # 7. Fixture team xG
132:     # ------------------------------
133: 
134:     # Backward-compatible GW1 behaviour:
135:     # if no explicit team_xg table is supplied,
136:     # build it from the bookmaker market.
137:     if team_xg is None:
138: 
139:         if market_odds is None:
140:             raise ValueError(
141:                 "Provide either market_odds "
142:                 "or a team_xg DataFrame."
143:             )
144: 
145:         fixture_team_xg = build_gw1_team_xg(
146:             market_odds
147:         )
148: 
149:     else:
150: 
151:         required = {
152:             "Team",
153:             "Team xG",
154:         }
155: 


In [77]:
for i in range(155, len(src)):
    print(f"{i + 1:03d}: {src[i]}")

156:         missing = (
157:             required
158:             - set(team_xg.columns)
159:         )
160: 
161:         if missing:
162:             raise ValueError(
163:                 "team_xg missing required columns: "
164:                 f"{sorted(missing)}"
165:             )
166: 
167:         fixture_team_xg = (
168:             team_xg[
169:                 [
170:                     "Team",
171:                     "Team xG",
172:                 ]
173:             ]
174:             .drop_duplicates("Team")
175:             .copy()
176:         )
177: 
178: 
179:     df = df.merge(
180:         fixture_team_xg,
181:         on="Team",
182:         how="left",
183:         validate="many_to_one",
184:     )
185: 
186:     if df["Team xG"].isna().any():
187: 
188:         missing_teams = (
189:             df.loc[
190:                 df["Team xG"].isna(),
191:                 "Team",
192:             ]
193:             .drop_duplicates()
194:             .tolist()
195

In [78]:
attack.groupby("Team").agg(
    Player_xG=("GW1 xG", "sum"),
    Market_xG=("Team xG", "first"),
).assign(
    Error=lambda x:
        x["Player_xG"]
        - x["Market_xG"]
)

,Player_xG,Market_xG,Error
Team,,,
Arsenal,2.652936,2.652936,0.000000e+00
Aston Villa,1.214704,1.214704,0.000000e+00
Bournemouth,0.919178,0.919178,0.000000e+00
Brentford,1.372034,1.372034,0.000000e+00
Brighton,1.452548,1.452548,-2.220446e-16
Chelsea,1.529191,1.529191,-2.220446e-16
Coventry City,0.528659,0.528659,0.000000e+00
Crystal Palace,1.005421,1.005421,0.000000e+00
Everton,1.304248,1.304248,0.000000e+00


In [79]:
from src.attack_projection import (
    build_attack_projection,
    build_attack_horizon,
)

attack_horizon = build_attack_horizon(
    minutes=minutes,
    attack_priors=model["Attack_Priors"],
    workbook_players=model["Players"],
    fixture_horizon=fixture_horizon,
    external_priors=external_priors,
    max_gw=6,
)

print(
    "Attack horizon:",
    attack_horizon.shape,
)

Attack horizon: (3594, 63)


In [80]:
attack_check = (
    attack_horizon
    .groupby(
        ["GW", "Team"]
    )
    .agg(
        Player_xG=("xG", "sum"),
        Fixture_xG=("Team xG", "first"),
    )
)

attack_check["Error"] = (
    attack_check["Player_xG"]
    - attack_check["Fixture_xG"]
)

print(
    "Max absolute reconciliation error:",
    attack_check["Error"]
    .abs()
    .max()
)

display(
    attack_check
)

Max absolute reconciliation error: 4.440892098500626e-16


Player_xG  Fixture_xG         Error
GW Team                                              
1  Arsenal         2.652936    2.652936  0.000000e+00
   Aston Villa     1.214704    1.214704  0.000000e+00
   Bournemouth     0.919178    0.919178  0.000000e+00
   Brentford       1.372034    1.372034  0.000000e+00
   Brighton        1.452548    1.452548 -2.220446e-16
...                     ...         ...           ...
6  Man Utd         1.545354    1.545354  0.000000e+00
   Newcastle       1.333560    1.333560 -2.220446e-16
   Nott'm Forest   1.026748    1.026748  0.000000e+00
   Spurs           0.848872    0.848872  1.110223e-16
   Sunderland      0.908415    0.908415  1.110223e-16

[120 rows x 3 columns]

In [81]:
watch = [
    "Haaland",
    "Gabriel",
    "B.Fernandes",
    "Mbeumo",
    "Tzolis",
]

xg_view = (
    attack_horizon[
        attack_horizon["Player"].isin(
            watch
        )
    ]
    .pivot_table(
        index=[
            "Player",
            "Team",
        ],
        columns="GW",
        values="xG",
    )
)

xg_view["GW1-6 xG"] = (
    xg_view.sum(axis=1)
)

display(
    xg_view.sort_values(
        "GW1-6 xG",
        ascending=False,
    )
)

,GW,1,2,3,4,5,6,GW1-6 xG
Player,Team,,,,,,,
Haaland,Man City,0.658363,0.472371,0.547930,0.443393,0.537192,0.432838,3.092087
Mbeumo,Man Utd,0.369739,0.283950,0.285419,0.244845,0.270313,0.287947,1.742213
B.Fernandes,Man Utd,0.319097,0.245059,0.246326,0.211309,0.233289,0.248508,1.503588
Tzolis,Arsenal,0.247219,0.136976,0.144607,0.138200,0.125201,0.153656,0.945859
Gabriel,Arsenal,0.132006,0.073140,0.077215,0.073794,0.066853,0.082047,0.505054


In [82]:
xa_view = (
    attack_horizon[
        attack_horizon["Player"].isin(
            watch
        )
    ]
    .pivot_table(
        index=[
            "Player",
            "Team",
        ],
        columns="GW",
        values="xA",
    )
)

xa_view["GW1-6 xA"] = (
    xa_view.sum(axis=1)
)

display(
    xa_view.sort_values(
        "GW1-6 xA",
        ascending=False,
    )
)

,GW,1,2,3,4,5,6,GW1-6 xA
Player,Team,,,,,,,
B.Fernandes,Man Utd,0.347922,0.267195,0.268577,0.230397,0.254362,0.270956,1.639409
Mbeumo,Man Utd,0.176875,0.135835,0.136538,0.117128,0.129311,0.137747,0.833434
Tzolis,Arsenal,0.207493,0.114965,0.121370,0.115992,0.105083,0.128965,0.793868
Haaland,Man City,0.073146,0.052482,0.060877,0.049263,0.059684,0.048090,0.343541
Gabriel,Arsenal,0.089454,0.049564,0.052325,0.050007,0.045303,0.055599,0.342253


In [83]:
import inspect
import src.xpts as xp

print("XPTS FUNCTIONS:")
for name in dir(xp):
    obj = getattr(xp, name)

    if callable(obj) and not name.startswith("_"):
        try:
            print(name, inspect.signature(obj))
        except Exception:
            pass

XPTS FUNCTIONS:
build_gw1_xpts (attack: pandas.DataFrame, appearance: pandas.DataFrame) -> pandas.DataFrame
build_xpts (attack: pandas.DataFrame, appearance: pandas.DataFrame = None) -> pandas.DataFrame
expected_conceded_deduction (lam, max_goals=12)


In [84]:
import inspect
import src.defensive_points as dp

print("DEFENSIVE FUNCTIONS:")
for name in dir(dp):
    obj = getattr(dp, name)

    if callable(obj) and not name.startswith("_"):
        try:
            print(name, inspect.signature(obj))
        except Exception:
            pass

DEFENSIVE FUNCTIONS:
build_defensive_priors (current_players: pandas.DataFrame, hist: pandas.DataFrame) -> pandas.DataFrame
build_defensive_xpts (start_probs, defensive_priors, defcon_calibrator=None)


In [85]:
print(
    inspect.getsource(
        xp.build_gw1_xpts
    )
)

print(
    inspect.getsource(
        dp.build_defensive_xpts
    )
)

def build_gw1_xpts(
    attack: pd.DataFrame,
    appearance: pd.DataFrame,
) -> pd.DataFrame:
    """
    Backwards-compatible wrapper for the old GW1 pipeline.
    """

    out = build_xpts(
        attack=attack,
        appearance=appearance,
    )

    # Preserve the name expected elsewhere
    # in the existing GW1 notebook.
    out["GW1 Base xPts"] = (
        out["Base xPts"]
    )

    return out

def build_defensive_xpts(
    start_probs,
    defensive_priors,
    defcon_calibrator=None,
):

    df = start_probs[
    [
        "Player ID",
        "FPL Pos",
        "Conditional Start Prob",
        "Availability Prob",
    ]
    ].merge(
        defensive_priors[
            [
                "Player ID",
                "P DC | Start",
                "P DC | Not Start",
                "Save Pts | Start",
                "Save Pts | Not Start",
            ]
        ],
        on="Player ID",
        how="left",
    )

    s = df["Conditional Start Prob"]
    a = df["Avail

In [86]:
import inspect
import src.xpts as xp

src = inspect.getsource(
    xp.build_gw1_xpts
).splitlines()

print("Length:", len(src))

for i in range(25, min(70, len(src))):
    print(f"{i + 1:03d}: {src[i]}")

Length: 20


In [87]:
for i in range(70, len(src)):
    print(f"{i + 1:03d}: {src[i]}")

In [88]:
import src.defensive_points as dp

src_def = inspect.getsource(
    dp.build_defensive_xpts
).splitlines()

print("Length:", len(src_def))

for i in range(0, min(60, len(src_def))):
    print(f"{i + 1:03d}: {src_def[i]}")

Length: 66
001: def build_defensive_xpts(
002:     start_probs,
003:     defensive_priors,
004:     defcon_calibrator=None,
005: ):
006: 
007:     df = start_probs[
008:     [
009:         "Player ID",
010:         "FPL Pos",
011:         "Conditional Start Prob",
012:         "Availability Prob",
013:     ]
014:     ].merge(
015:         defensive_priors[
016:             [
017:                 "Player ID",
018:                 "P DC | Start",
019:                 "P DC | Not Start",
020:                 "Save Pts | Start",
021:                 "Save Pts | Not Start",
022:             ]
023:         ],
024:         on="Player ID",
025:         how="left",
026:     )
027: 
028:     s = df["Conditional Start Prob"]
029:     a = df["Availability Prob"]
030: 
031:     df["P DC Start Used"] = df["P DC | Start"]
032: 
033:     if defcon_calibrator is not None:
034: 
035:         from src.defcon_calibration import (
036:             calibrate_defender_prob,
037:         )
038: 
039:         

In [89]:
for i in range(60, len(src_def)):
    print(f"{i + 1:03d}: {src_def[i]}")

061:     df["xPts Saves"] = a * (
062:         s * df["Save Pts | Start"]
063:         + (1 - s) * df["Save Pts | Not Start"]
064:     )
065: 
066:     return df


In [90]:
for i in range(23, 60):
    print(f"{i + 1:03d}: {src_def[i]}")

024:         on="Player ID",
025:         how="left",
026:     )
027: 
028:     s = df["Conditional Start Prob"]
029:     a = df["Availability Prob"]
030: 
031:     df["P DC Start Used"] = df["P DC | Start"]
032: 
033:     if defcon_calibrator is not None:
034: 
035:         from src.defcon_calibration import (
036:             calibrate_defender_prob,
037:         )
038: 
039:         mask = df["FPL Pos"] == "DEF"
040: 
041:         df.loc[
042:             mask,
043:             "P DC Start Used"
044:         ] = calibrate_defender_prob(
045:             df.loc[
046:                 mask,
047:                 "P DC | Start"
048:             ],
049:             defcon_calibrator,
050:         )
051: 
052:     df["P Defensive Return"] = a * (
053:         s * df["P DC Start Used"]
054:         + (1 - s) * df["P DC | Not Start"]
055:     )
056: 
057:     df["xPts DefCon"] = (
058:         2 * df["P Defensive Return"]
059:     )
060: 


In [91]:
from src.xpts import build_xpts

base_horizon = build_xpts(
    attack_horizon,
    appearance,
)

print(
    "Base horizon:",
    base_horizon.shape,
)

Base horizon: (3594, 74)


In [92]:
defensive_xpts = build_defensive_xpts(
    start_probs,
    defensive_priors,
    defcon_calibrator=dc_calibrator,
)


In [93]:
xpts_horizon = base_horizon.merge(
    defensive_xpts,
    on="Player ID",
    how="left",
    validate="many_to_one",
)

xpts_horizon[
    [
        "xPts DefCon",
        "xPts Saves",
    ]
] = (
    xpts_horizon[
        [
            "xPts DefCon",
            "xPts Saves",
        ]
    ]
    .fillna(0.0)
)

xpts_horizon["xPts Model"] = (
    xpts_horizon["Base xPts"]
    + xpts_horizon["xPts DefCon"]
    + xpts_horizon["xPts Saves"]
)

print(
    "Full xPts horizon:",
    xpts_horizon.shape,
)

Full xPts horizon: (3594, 86)


In [94]:
pts_matrix = (
    xpts_horizon
    .pivot_table(
        index=[
            "Player ID",
            "Player",
            "Team",
            "FPL Pos_x",
        ],
        columns="GW",
        values="xPts Model",
    )
)

pts_matrix.columns = [
    f"GW{int(gw)}"
    for gw in pts_matrix.columns
]

pts_matrix["GW1-6 xPts"] = (
    pts_matrix[
        [
            "GW1",
            "GW2",
            "GW3",
            "GW4",
            "GW5",
            "GW6",
        ]
    ].sum(axis=1)
)

pts_matrix = (
    pts_matrix
    .reset_index()
    .merge(
        current_players[
            [
                "Player ID",
                "Current £m",
            ]
        ],
        on="Player ID",
        how="left",
    )
)

display(
    pts_matrix.sort_values(
        "GW1-6 xPts",
        ascending=False,
    ).head(40)
)

,Player ID,Player,Team,FPL Pos_x,GW1,GW2,GW3,GW4,GW5,GW6,GW1-6 xPts,Current £m
3,4,Gabriel,Arsenal,DEF,5.562764,4.801300,4.506659,5.081328,4.492584,4.837974,29.282610,8.0
425,426,B.Fernandes,Man Utd,MID,5.322800,4.548618,4.534743,4.143698,4.415828,4.629204,27.594892,12.0
7,8,Calafiori,Arsenal,DEF,5.242596,4.434315,4.185686,4.681868,4.148628,4.487645,27.180737,5.5
480,481,Anderson,Man City,MID,4.802111,4.351385,4.562006,4.259526,4.632046,4.258121,26.865194,6.5
10,11,Mosquera,Arsenal,DEF,4.814250,4.287390,4.004911,4.542743,4.022335,4.294959,25.966588,5.5
12,13,Rice,Arsenal,MID,4.831633,4.017323,4.001185,4.084915,3.882981,4.123718,24.941754,7.5
426,427,Mbeumo,Man Utd,MID,4.820996,4.107419,4.092556,3.730220,3.984281,4.185100,24.920572,8.0
410,411,Haaland,Man City,FWD,4.743804,3.937843,4.265268,3.812275,4.218733,3.766538,24.744462,15.5
9,10,White,Arsenal,DEF,4.637885,4.035969,3.779791,4.275767,3.776722,4.059643,24.565776,5.5
70,71,Cook,Bournemouth,MID,3.724171,4.273052,4.047655,4.166647,4.058598,4.020162,24.290285,5.0


In [95]:
display(
    pts_matrix[
        pts_matrix["Player"].isin(
            [
                "Haaland",
                "B.Fernandes",
                "Mbeumo",
                "Gabriel",
                "Tzolis",
                "Isak",
                "João Pedro",
                "Gvardiol",
                "Pedro Porro",
            ]
        )
    ].sort_values(
        "GW1-6 xPts",
        ascending=False,
    )
)

,Player ID,Player,Team,FPL Pos_x,GW1,GW2,GW3,GW4,GW5,GW6,GW1-6 xPts,Current £m
3,4,Gabriel,Arsenal,DEF,5.562764,4.801300,4.506659,5.081328,4.492584,4.837974,29.282610,8.0
425,426,B.Fernandes,Man Utd,MID,5.322800,4.548618,4.534743,4.143698,4.415828,4.629204,27.594892,12.0
426,427,Mbeumo,Man Utd,MID,4.820996,4.107419,4.092556,3.730220,3.984281,4.185100,24.920572,8.0
410,411,Haaland,Man City,FWD,4.743804,3.937843,4.265268,3.812275,4.218733,3.766538,24.744462,15.5
556,557,Tzolis,Arsenal,MID,4.518302,3.631703,3.625766,3.695901,3.493035,3.750240,22.714946,6.5
390,391,Gvardiol,Man City,DEF,4.041148,3.400744,3.796354,3.185527,4.197468,3.272565,21.893805,5.5
164,165,João Pedro,Chelsea,FWD,3.286912,3.239732,2.609674,4.291655,3.261208,3.446639,20.135821,7.5
378,379,Isak,Liverpool,FWD,3.434564,3.437473,3.174190,3.335212,3.301030,3.092862,19.775331,9.0
498,499,Pedro Porro,Spurs,DEF,1.158589,1.150763,1.200680,1.239063,1.220759,1.046851,7.016704,5.5


In [96]:
component_cols = [
    "xPts Appearance",
    "xPts Goals",
    "xPts Assists",
    "xPts Clean Sheet",
    "xPts Goals Conceded",
    "xPts DefCon",
    "xPts Saves",
    "xPts Model",
]

pos_col = (
    "FPL Pos"
    if "FPL Pos" in xpts_horizon.columns
    else "FPL Pos_x"
)

component_audit = (
    xpts_horizon
    .groupby(
        [
            "Player ID",
            "Player",
            "Team",
            pos_col,
        ]
    )[component_cols]
    .sum()
    .reset_index()
    .sort_values(
        "xPts Model",
        ascending=False,
    )
)

display(
    component_audit.head(30)
)

,Player ID,Player,Team,FPL Pos_x,xPts Appearance,xPts Goals,xPts Assists,xPts Clean Sheet,xPts Goals Conceded,xPts DefCon,xPts Saves,xPts Model
3,4,Gabriel,Arsenal,DEF,11.400597,3.030323,1.026759,11.731335,-0.798406,2.892002,0.00000,29.282610
425,426,B.Fernandes,Man Utd,MID,11.356810,7.517941,4.918228,2.105443,0.000000,1.696469,0.00000,27.594892
7,8,Calafiori,Arsenal,DEF,10.777575,4.015668,0.743819,10.474288,-0.615935,1.785322,0.00000,27.180737
480,481,Anderson,Man City,MID,11.479328,3.505130,2.539108,1.980642,0.000000,7.360986,0.00000,26.865194
10,11,Mosquera,Arsenal,DEF,11.015490,1.253548,0.991864,10.853556,-0.693300,2.545430,0.00000,25.966588
12,13,Rice,Arsenal,MID,11.372007,3.160038,3.292342,2.913299,0.000000,4.204067,0.00000,24.941754
426,427,Mbeumo,Man Utd,MID,11.381646,8.711063,2.500303,2.101615,0.000000,0.225945,0.00000,24.920572
410,411,Haaland,Man City,FWD,11.328696,12.368348,1.030624,0.000000,0.000000,0.016794,0.00000,24.744462
9,10,White,Arsenal,DEF,10.621724,1.908815,1.129960,10.196493,-0.618707,1.327491,0.00000,24.565776
70,71,Cook,Bournemouth,MID,11.351553,2.708943,1.673876,1.209368,0.000000,7.346545,0.00000,24.290285


In [97]:
display(
    component_audit[
        component_audit["Player"].isin(
            [
                "Gabriel",
                "Anderson",
                "Haaland",
                "B.Fernandes",
                "Mbeumo",
                "Pedro Porro",
            ]
        )
    ]
)

,Player ID,Player,Team,FPL Pos_x,xPts Appearance,xPts Goals,xPts Assists,xPts Clean Sheet,xPts Goals Conceded,xPts DefCon,xPts Saves,xPts Model
3,4,Gabriel,Arsenal,DEF,11.400597,3.030323,1.026759,11.731335,-0.798406,2.892002,0.0,29.282610
425,426,B.Fernandes,Man Utd,MID,11.356810,7.517941,4.918228,2.105443,0.000000,1.696469,0.0,27.594892
480,481,Anderson,Man City,MID,11.479328,3.505130,2.539108,1.980642,0.000000,7.360986,0.0,26.865194
426,427,Mbeumo,Man Utd,MID,11.381646,8.711063,2.500303,2.101615,0.000000,0.225945,0.0,24.920572
410,411,Haaland,Man City,FWD,11.328696,12.368348,1.030624,0.000000,0.000000,0.016794,0.0,24.744462
498,499,Pedro Porro,Spurs,DEF,4.134259,0.339401,0.536835,1.632362,-0.282664,0.656511,0.0,7.016704


In [98]:
display(
    minutes[
        minutes["Player"].isin(
            [
                "Gabriel",
                "Anderson",
                "Haaland",
                "B.Fernandes",
                "Pedro Porro",
            ]
        )
    ][
        [
            "Player",
            "Team",
            "Start Prob",
            "Effective Mins",
        ]
    ]
)

,Player,Team,Start Prob,Effective Mins
3,Gabriel,Arsenal,0.947801,84.665231
448,Haaland,Man City,0.955595,81.726370
449,Anderson,Man City,0.955595,84.337442
468,B.Fernandes,Man Utd,0.946306,83.315151
545,Pedro Porro,Spurs,0.230993,24.679182


In [99]:
def_audit = defensive_xpts.merge(
    current_players[
        [
            "Player ID",
            "Player",
            "Team",
        ]
    ],
    on="Player ID",
    how="left",
)

wanted = [
    c for c in [
        "Player",
        "Team",
        "FPL Pos",
        "P DC | Start",
        "P DC Start Used",
        "P Defensive Return",
        "xPts DefCon",
        "xPts Saves",
    ]
    if c in def_audit.columns
]

display(
    def_audit[
        def_audit["Player"].isin(
            [
                "Gabriel",
                "Anderson",
                "Rice",
                "Cook",
                "Ampadu",
                "Haaland",
                "B.Fernandes",
                "Mbeumo",
                "Pedro Porro",
            ]
        )
    ][wanted]
)

,Player,Team,FPL Pos,P DC | Start,P DC Start Used,P Defensive Return,xPts DefCon,xPts Saves
3,Gabriel,Arsenal,DEF,0.418703,0.254199,0.241000,0.482000,0.0
12,Rice,Arsenal,MID,0.369613,0.369613,0.350339,0.700678,0.0
72,Cook,Bournemouth,MID,0.639639,0.639639,0.612212,1.224424,0.0
373,Ampadu,Leeds,MID,0.517781,0.517781,0.498236,0.996471,0.0
448,Haaland,Man City,FWD,0.001465,0.001465,0.001399,0.002799,0.0
449,Anderson,Man City,MID,0.641892,0.641892,0.613415,1.226831,0.0
468,B.Fernandes,Man Utd,MID,0.149360,0.149360,0.141372,0.282745,0.0
469,Mbeumo,Man Utd,MID,0.019873,0.019873,0.018829,0.037657,0.0
545,Pedro Porro,Spurs,DEF,0.382815,0.231704,0.054709,0.109419,0.0


In [100]:
print(appearance_priors.columns.tolist())

display(
    appearance_priors[
        appearance_priors["Player"].isin(
            [
                "Pedro Porro",
                "Haaland",
                "Gabriel",
                "B.Fernandes",
                "Saka",
                "Anderson",
                "Foden",
            ]
        )
    ]
)

['Player ID', 'Code', 'Player', 'Team', 'FPL Pos', 'player_code', 'Starts', 'Start60', 'player_code_bench', 'NonStarts', 'BenchApps', 'Bench60', 'P60 If Start', 'P(App | Not Start)', 'P60 If Not Start']


,Player ID,Code,Player,Team,FPL Pos,player_code,Starts,Start60,player_code_bench,NonStarts,BenchApps,Bench60,P60 If Start,P(App | Not Start),P60 If Not Start
3,4,226597,Gabriel,Arsenal,DEF,226597.0,28.0,28.0,226597.0,4.0,2.0,0.0,0.989497,0.274067,0.002807
11,12,223340,Saka,Arsenal,MID,223340.0,24.0,23.0,223340.0,10.0,5.0,0.0,0.946845,0.411729,0.001982
435,398,209244,Foden,Man City,MID,209244.0,23.0,22.0,209244.0,15.0,9.0,0.0,0.945131,0.489383,0.001585
448,411,223094,Haaland,Man City,FWD,223094.0,33.0,32.0,223094.0,5.0,1.0,0.0,0.959990,0.339953,0.001418
449,481,215379,Anderson,Man City,MID,215379.0,36.0,36.0,215379.0,1.0,1.0,0.0,0.984069,0.384961,0.003603
468,426,141746,B.Fernandes,Man Utd,MID,141746.0,34.0,34.0,141746.0,1.0,0.0,0.0,0.983311,0.294052,0.003603
545,499,441164,Pedro Porro,Spurs,DEF,441164.0,31.0,31.0,441164.0,3.0,2.0,0.0,0.990305,0.295149,0.003023


In [101]:
print(minutes.columns.tolist())

display(
    minutes[
        minutes["Player"].isin(
            [
                "Pedro Porro",
                "Haaland",
                "Gabriel",
                "B.Fernandes",
                "Saka",
            ]
        )
    ]
)

['Player ID', 'Code', 'Player', 'Team', 'FPL Pos', 'Status', 'Chance Play Next', 'News', 'FFScout', 'RotoWire', 'NMA', 'Lineup Votes', 'Lineup Consensus', 'Availability Prob', 'Lineup Evidence', 'Conditional Start Prob', 'Source Start Prob', 'Start Prob', 'Mins If Start', 'Expected Mins If Not Start', 'Start_Sample', 'Bench_Sample', 'Model Expected Mins', 'User Mins Override', 'Effective Mins']


,Player ID,Code,Player,Team,FPL Pos,Status,Chance Play Next,News,FFScout,RotoWire,...,Conditional Start Prob,Source Start Prob,Start Prob,Mins If Start,Expected Mins If Not Start,Start_Sample,Bench_Sample,Model Expected Mins,User Mins Override,Effective Mins
3,4,226597,Gabriel,Arsenal,DEF,a,NaN,,1.0,1.0,...,0.947801,0.947801,0.947801,88.914663,7.506385,28.0,4.0,84.665231,<NA>,84.665231
11,12,223340,Saka,Arsenal,MID,a,NaN,,0.0,1.0,...,0.249568,0.249568,0.249568,83.677847,10.433426,24.0,10.0,28.712871,<NA>,28.712871
448,411,223094,Haaland,Man City,FWD,a,NaN,,1.0,1.0,...,0.955595,0.955595,0.955595,85.259496,5.693071,33.0,5.0,81.726370,<NA>,81.726370
468,426,141746,B.Fernandes,Man Utd,MID,a,NaN,,1.0,1.0,...,0.946306,0.946306,0.946306,87.734809,5.422407,34.0,1.0,83.315151,<NA>,83.315151
545,499,441164,Pedro Porro,Spurs,DEF,a,NaN,,0.0,1.0,...,0.230993,0.230993,0.230993,87.143997,5.916056,31.0,3.0,24.679182,<NA>,24.679182


In [102]:
from src.horizon_minutes import (
    build_minutes_horizon,
)

minutes_horizon = build_minutes_horizon(
    minutes,
    appearance_priors,
    max_gw=6,
)

appearance_horizon = minutes_horizon[
    [
        "Player ID",
        "GW",
        "Appearance Prob",
        "P60",
        "Expected Appearance Pts",
    ]
].copy()

In [103]:
display(
    minutes_horizon[
        minutes_horizon["Player"].isin(
            [
                "Pedro Porro",
                "Saka",
                "Haaland",
                "Gabriel",
                "B.Fernandes",
            ]
        )
    ][
        [
            "GW",
            "Player",
            "Availability Prob",
            "Historical Start Prob",
            "Conditional Start Prob",
            "Start Prob",
            "Effective Mins",
        ]
    ]
)

,GW,Player,Availability Prob,Historical Start Prob,Conditional Start Prob,Start Prob,Effective Mins
3,1,Gabriel,1.0,0.875000,0.947801,0.947801,84.665231
11,1,Saka,1.0,0.705882,0.249568,0.249568,28.712871
448,1,Haaland,1.0,0.868421,0.955595,0.955595,81.726370
468,1,B.Fernandes,1.0,0.971429,0.946306,0.946306,83.315151
545,1,Pedro Porro,1.0,0.911765,0.230993,0.230993,24.679182
602,2,Gabriel,1.0,0.875000,0.921081,0.921081,82.489965
610,2,Saka,1.0,0.705882,0.370208,0.370208,37.549102
1047,2,Haaland,1.0,0.868421,0.918985,0.918985,78.813422
1067,2,B.Fernandes,1.0,0.971429,0.945808,0.945808,83.274148
1144,2,Pedro Porro,1.0,0.911765,0.344144,0.344144,33.870129


In [104]:
print(
    minutes_horizon.groupby(
        ["GW", "Team"]
    )["Start Prob"].sum()
    .sub(11)
    .abs()
    .max()
)

1.7319479184152442e-12


In [105]:
attack_horizon = build_attack_horizon(
    minutes=minutes_horizon,
    attack_priors=model["Attack_Priors"],
    workbook_players=model["Players"],
    fixture_horizon=fixture_horizon,
    external_priors=external_priors,
    max_gw=6,
)

In [106]:
base_horizon = build_xpts(
    attack_horizon,
    appearance_horizon,
)

In [107]:
def_parts = []

for gw in range(1, 7):

    gw_starts = minutes_horizon[
        minutes_horizon["GW"] == gw
    ].copy()

    d = build_defensive_xpts(
        gw_starts,
        defensive_priors,
        dc_calibrator,
    )

    d["GW"] = gw

    def_parts.append(d)

defensive_horizon = pd.concat(
    def_parts,
    ignore_index=True,
)

In [108]:
xpts_horizon = base_horizon.merge(
    defensive_horizon[
        [
            "Player ID",
            "GW",
            "xPts DefCon",
            "xPts Saves",
        ]
    ],
    on=[
        "Player ID",
        "GW",
    ],
    how="left",
    validate="one_to_one",
)

xpts_horizon[
    ["xPts DefCon", "xPts Saves"]
] = (
    xpts_horizon[
        ["xPts DefCon", "xPts Saves"]
    ]
    .fillna(0.0)
)

xpts_horizon["xPts Model"] = (
    xpts_horizon["Base xPts"]
    + xpts_horizon["xPts DefCon"]
    + xpts_horizon["xPts Saves"]
)

In [109]:
base_horizon = build_xpts(
    attack_horizon,
    appearance_horizon,
)

print(
    "Base horizon:",
    base_horizon.shape,
)

Base horizon: (3594, 72)


In [110]:
display(
    base_horizon[
        base_horizon["Player"].isin(
            [
                "Pedro Porro",
                "Saka",
                "Haaland",
            ]
        )
    ][
        [
            "GW",
            "Player",
            "Effective Mins",
            "Appearance Prob",
            "P60",
            "xG",
            "xA",
            "Base xPts",
        ]
    ].sort_values(
        ["Player", "GW"]
    )
)

,GW,Player,Effective Mins,Appearance Prob,P60,xG,xA,Base xPts
448,1,Haaland,81.726370,0.970691,0.917425,0.658363,0.073146,4.741005
1047,2,Haaland,78.813422,0.946526,0.882332,0.454595,0.050507,3.798761
1646,3,Haaland,76.534612,0.927622,0.854878,0.511979,0.056883,4.001063
2245,4,Haaland,74.548833,0.911149,0.830954,0.403888,0.044873,3.492277
2844,5,Haaland,73.040623,0.898638,0.812784,0.480021,0.053332,3.791504
3443,6,Haaland,72.215863,0.891796,0.802848,0.382747,0.042524,3.353204
545,1,Pedro Porro,24.679182,0.457965,0.231078,0.011726,0.037096,1.049170
1144,2,Pedro Porro,33.870129,0.537719,0.342789,0.012265,0.038800,1.376126
1743,3,Pedro Porro,40.532491,0.595531,0.423767,0.015201,0.048088,1.711564
2342,4,Pedro Porro,45.747711,0.640786,0.487155,0.018685,0.059108,1.997575


In [111]:
from src.horizon_optimizer import (
    optimize_horizon_squad,
)

opt = optimize_horizon_squad(
    xpts_horizon=xpts_horizon,
    current_players=current_players,
    budget=100.0,
    max_gw=6,
    gw_decay=0.90,
    bench_weight=0.12,
)

model_squad = opt["squad"]

print(
    "Cost:",
    opt["total_cost"],
)

print(
    "Objective:",
    opt["objective"],
)

display(
    model_squad[
        [
            "Player",
            "Team",
            "FPL Pos",
            "Current £m",
            "GW1",
            "GW2",
            "GW3",
            "GW4",
            "GW5",
            "GW6",
            "6GW xPts",
            "Start GW1",
            "Captain GW1",
        ]
    ]
)

Cost: 100.0
Objective: 242.49090737028172


,Player,Team,FPL Pos,Current £m,GW1,GW2,GW3,GW4,GW5,GW6,6GW xPts,Start GW1,Captain GW1
3,Gabriel,Arsenal,DEF,8.0,5.562764,4.655491,4.263780,4.706900,4.096841,4.361475,27.647251,True,True
250,Tarkowski,Everton,DEF,6.0,4.290076,3.366498,3.403994,4.087006,3.988712,4.558596,23.694882,True,False
392,Virgil,Liverpool,DEF,6.5,4.056973,4.350820,4.034892,4.200980,3.613089,3.428853,23.685607,True,False
536,Diomande,Nott'm Forest,DEF,5.5,4.515699,3.380738,4.347152,3.830219,3.947308,3.464796,23.485913,True,False
487,Thiaw,Newcastle,DEF,5.0,3.164591,4.059447,3.569945,3.526781,4.682711,3.532197,22.535671,False,False
107,Thiago,Brentford,FWD,8.0,3.519332,3.534663,3.626589,3.555542,3.559980,3.474999,21.271106,True,False
206,Simms,Coventry City,FWD,5.0,2.419734,4.033477,2.868520,3.057266,3.127444,3.093351,18.599792,False,False
346,Emersonn,Ipswich Town,FWD,5.5,2.972403,2.856765,2.919259,2.924849,3.015426,3.049217,17.737919,False,False
0,Raya,Arsenal,GK,6.0,4.316061,4.004771,3.661118,4.245589,3.699955,3.913583,23.841078,True,False
111,Verbruggen,Brighton,GK,4.5,3.243161,2.895544,3.425869,3.340293,3.100419,3.839924,19.845210,False,False


In [112]:
display(
    model_squad[
        model_squad["Start GW1"]
    ][
        [
            "Player",
            "Team",
            "FPL Pos",
            "Current £m",
            "GW1",
            "Captain GW1",
        ]
    ].sort_values(
        "GW1",
        ascending=False,
    )
)

,Player,Team,FPL Pos,Current £m,GW1,Captain GW1
3,Gabriel,Arsenal,DEF,8.0,5.562764,True
468,B.Fernandes,Man Utd,MID,12.0,5.322800,False
12,Rice,Arsenal,MID,7.5,4.831633,False
469,Mbeumo,Man Utd,MID,8.0,4.820996,False
449,Anderson,Man City,MID,6.5,4.802111,False
536,Diomande,Nott'm Forest,DEF,5.5,4.515699,False
0,Raya,Arsenal,GK,6.0,4.316061,False
250,Tarkowski,Everton,DEF,6.0,4.290076,False
392,Virgil,Liverpool,DEF,6.5,4.056973,False
258,Ndiaye,Everton,MID,6.0,3.889228,False


In [113]:
opt_haaland = optimize_horizon_squad(
    xpts_horizon=xpts_horizon,
    current_players=current_players,
    budget=100.0,
    max_gw=6,
    gw_decay=0.90,
    bench_weight=0.12,
    force_players=["Haaland"],
)

print("Unrestricted:", opt["objective"])
print("Haaland forced:", opt_haaland["objective"])
print(
    "Cost of forcing Haaland:",
    opt["objective"]
    - opt_haaland["objective"]
)

display(
    opt_haaland["squad"][
        [
            "Player",
            "Team",
            "FPL Pos",
            "Current £m",
            "GW1",
            "GW2",
            "GW3",
            "GW4",
            "GW5",
            "GW6",
            "6GW xPts",
            "Start GW1",
            "Captain GW1",
        ]
    ]
)

Unrestricted: 242.49090737028172
Haaland forced: 239.30868168603334
Cost of forcing Haaland: 3.1822256842483796


,Player,Team,FPL Pos,Current £m,GW1,GW2,GW3,GW4,GW5,GW6,6GW xPts,Start GW1,Captain GW1
3,Gabriel,Arsenal,DEF,8.0,5.562764,4.655491,4.263780,4.706900,4.096841,4.361475,27.647251,True,True
250,Tarkowski,Everton,DEF,6.0,4.290076,3.366498,3.403994,4.087006,3.988712,4.558596,23.694882,True,False
392,Virgil,Liverpool,DEF,6.5,4.056973,4.350820,4.034892,4.200980,3.613089,3.428853,23.685607,True,False
536,Diomande,Nott'm Forest,DEF,5.5,4.515699,3.380738,4.347152,3.830219,3.947308,3.464796,23.485913,True,False
487,Thiaw,Newcastle,DEF,5.0,3.164591,4.059447,3.569945,3.526781,4.682711,3.532197,22.535671,False,False
448,Haaland,Man City,FWD,15.5,4.743804,3.801453,4.003671,3.494812,3.793983,3.355653,23.193375,True,False
206,Simms,Coventry City,FWD,5.0,2.419734,4.033477,2.868520,3.057266,3.127444,3.093351,18.599792,False,False
351,Walle Egeli,Ipswich Town,FWD,4.5,2.127757,2.091031,2.146182,2.162267,2.226490,2.251819,13.005546,False,False
0,Raya,Arsenal,GK,6.0,4.316061,4.004771,3.661118,4.245589,3.699955,3.913583,23.841078,True,False
111,Verbruggen,Brighton,GK,4.5,3.243161,2.895544,3.425869,3.340293,3.100419,3.839924,19.845210,False,False


In [114]:
og_squad = opt["squad"].copy()

gw1_bench = og_squad[
    ~og_squad["Start GW1"]
].copy()

display(
    gw1_bench[
        [
            "Player",
            "Team",
            "FPL Pos",
            "Current £m",
            "GW1",
        ]
    ].sort_values(
        "GW1",
        ascending=False,
    )
)

,Player,Team,FPL Pos,Current £m,GW1
111,Verbruggen,Brighton,GK,4.5,3.243161
487,Thiaw,Newcastle,DEF,5.0,3.164591
346,Emersonn,Ipswich Town,FWD,5.5,2.972403
206,Simms,Coventry City,FWD,5.0,2.419734


In [116]:
bench_gk = gw1_bench[
    gw1_bench["FPL Pos"] == "GK"
]

bench_outfield = (
    gw1_bench[
        gw1_bench["FPL Pos"] != "GK"
    ]
    .sort_values(
        "GW1",
        ascending=False,
    )
)

print("BENCH GK")
display(
    bench_gk[
        ["Player", "Team", "GW1"]
    ]
)

print("1st / 2nd / 3rd SUB")
display(
    bench_outfield[
        ["Player", "Team", "FPL Pos", "GW1"]
    ]
)

BENCH GK


,Player,Team,GW1
111,Verbruggen,Brighton,3.243161


1st / 2nd / 3rd SUB


,Player,Team,FPL Pos,GW1
487,Thiaw,Newcastle,DEF,3.164591
346,Emersonn,Ipswich Town,FWD,2.972403
206,Simms,Coventry City,FWD,2.419734


In [117]:
opt_no_bench = optimize_horizon_squad(
    xpts_horizon=xpts_horizon,
    current_players=current_players,
    budget=100.0,
    max_gw=6,
    gw_decay=0.90,
    bench_weight=0.0,
)

display(
    opt_no_bench["squad"][
        [
            "Player",
            "Team",
            "FPL Pos",
            "Current £m",
            "6GW xPts",
            "Start GW1",
        ]
    ]
)

,Player,Team,FPL Pos,Current £m,6GW xPts,Start GW1
3,Gabriel,Arsenal,DEF,8.0,27.647251,True
250,Tarkowski,Everton,DEF,6.0,23.694882,True
392,Virgil,Liverpool,DEF,6.5,23.685607,True
536,Diomande,Nott'm Forest,DEF,5.5,23.485913,True
487,Thiaw,Newcastle,DEF,5.0,22.535671,False
107,Thiago,Brentford,FWD,8.0,21.271106,True
206,Simms,Coventry City,FWD,5.0,18.599792,False
288,Kusi-Asare,Fulham,FWD,4.5,2.752020,False
0,Raya,Arsenal,GK,6.0,23.841078,True
388,Woodman,Liverpool,GK,4.0,1.228112,False


In [118]:
stress_xpts = xpts_horizon.copy()

stress_xpts["xPts Model"] = (
    stress_xpts["xPts Model"]
    - 0.50 * stress_xpts["xPts DefCon"]
)

opt_half_dc = optimize_horizon_squad(
    xpts_horizon=stress_xpts,
    current_players=current_players,
    budget=100.0,
    max_gw=6,
    gw_decay=0.90,
    bench_weight=0.12,
)

display(
    opt_half_dc["squad"][
        [
            "Player",
            "Team",
            "FPL Pos",
            "Current £m",
            "6GW xPts",
            "Start GW1",
        ]
    ]
)

,Player,Team,FPL Pos,Current £m,6GW xPts,Start GW1
3,Gabriel,Arsenal,DEF,8.0,26.281259,True
536,Diomande,Nott'm Forest,DEF,5.5,22.307374,True
392,Virgil,Liverpool,DEF,6.5,21.996345,True
250,Tarkowski,Everton,DEF,6.0,21.516624,True
7,Calafiori,Arsenal,DEF,5.5,21.158742,True
107,Thiago,Brentford,FWD,8.0,21.120945,True
206,Simms,Coventry City,FWD,5.0,18.522862,False
380,Calvert-Lewin,Leeds,FWD,6.0,18.401635,False
0,Raya,Arsenal,GK,6.0,23.841078,True
508,Horníček,Newcastle,GK,5.0,19.719453,False
